# Free Energy Surface & Conformational Transition Analysis
## A10 (Ac-Ala₁₀-NHMe) and A8Q2 (Ac-Ala₈Gln₂-NHMe) — Post-Production Analysis

---

## Purpose and scientific context

This notebook analyses the **1 µs NVT production trajectories** generated by `A10_Prod_Run.ipynb`
and `A8Q2_Prod_Run.ipynb`. Each simulation is started from two independent initial conformations
(α-helix and fully-extended), yielding **four trajectories total** per peptide variant (two for A10,
two for A8Q2).

The overarching goal is a **CV-transfer study**: collective variables (TICA/SPIB slow modes) learned
on A10 will be transferred to A8Q2, and their ability to correctly reproduce the FES and kinetics
of A8Q2 will be tested. This notebook establishes the **ground-truth reference** for both peptides
from unbiased MD, against which any accelerated or CV-transferred model must be validated.

Starting from two independent conformations per peptide is essential: if a single 1 µs trajectory
starting from the helix gives the same FES and kinetics as one starting from the extended state, it
constitutes strong convergence evidence (the ergodic hypothesis is satisfied on this timescale).

---

## What is computed and why

### Part 1 — Convergence
| What | Why |
|------|-----|
| Cumulative helix fraction f(t) | Confirms the time average has stabilised; a still-drifting trace means the 1 µs run has not sampled enough transitions to be meaningful |
| Block Standard Error (BSE) | Estimates the true statistical uncertainty of the mean helix fraction; the BSE rises with block size and plateaus once blocks are longer than the correlation time — the plateau value is the true uncertainty (Grossfield & Zuckerman 2009) |
| Split-trajectory FES (1st vs 2nd half) | Shape-comparison of the FES in two halves; if the FES looks the same in both halves the simulation has reversibly sampled both basins |

### Part 2 — Order parameters (per-frame observables)
| What | Why |
|------|-----|
| Backbone Cα RMSD to ideal helix | Scalar that captures overall structural similarity to the helix; serves as the primary reaction coordinate because it reflects global shape, not just one bond |
| N_HB: number of backbone i→(i+4) H-bonds | Directly observable by IR (amide I) and indirectly by NMR (chemical shifts); correlates almost perfectly with helix content but is more sensitive to partial helix formation than RMSD |
| DSSP per-residue assignment | Algorithm-agnostic, residue-level secondary structure assignment based on electrostatic hydrogen bond energies (Kabsch-Sander); provides an independent third measure that does not use any threshold the analyst chose |

### Part 3 — Conformational populations
| What | Why |
|------|-----|
| Global helix fraction f_helix | The most commonly reported scalar in experimental and computational helix-coil work; directly compared to CD experiments |
| Per-residue helix fraction | Helix propensity varies along the chain; terminal fraying is expected at positions 1 and 10; Gln residues at positions 3 and 8 (A8Q2 only) should show reduced propensity relative to Ala |
| Per-bond H-bond occupancy | Shows *which specific* backbone H-bonds are stable; the pattern (e.g., central bonds more occupied than terminal bonds) confirms the peptide adopts a genuine α-helical register rather than a 3₁₀-helix or other structure |

### Part 4 — Free Energy Surfaces
| What | Why |
|------|-----|
| 2D FES: RMSD × N_HB | Two-dimensional probability landscape shows the helix basin (low RMSD, high N_HB) and the coil basin (high RMSD, low N_HB) as minima separated by a barrier; the barrier height directly gives ΔG‡ |
| 1D FES along N_HB | Most direct comparison with experimental CD (f_helix) and NMR data; the two-state nature (two minima separated by a barrier) justifies the two-state kinetic model used in Part 6 |
| Per-residue Ramachandran PMF | Position-specific conformational preferences; the depth of the α-helical basin (φ≈−62°, ψ≈−41°) at each residue quantifies how strongly that position drives or resists helix formation |

### Part 5 — Two-state classification (triple-criterion)
Three physically independent criteria are used simultaneously. Agreement among all three proves the
state assignment is not an artefact of the particular threshold or order parameter chosen.

| Criterion | Physical basis | Threshold | Reference |
|-----------|---------------|-----------|----------|
| RMSD to helix | Global backbone shape | < 2.0 Å (helix), > 3.0 Å (coil) | Kuczera 2021 |
| H-bond fraction | Electrostatic signature | ≥ 50% of max N_HB | Kuczera 2021 |
| DSSP | Energy-based algorithm | ≥ 50% residues = 'H' | Kabsch-Sander 1983 |

Hysteresis is applied to the RMSD criterion only: frames in the transition zone (2.0–3.0 Å) inherit
the state of the previous frame. Without hysteresis, frames hovering near a threshold generate
spurious 'flickering' transitions that inflate N_fold and N_unfold by factors of 2–10.

### Part 6 — Kinetics
| What | Why | Method |
|------|-----|--------|
| Folding/unfolding rates k_f, k_u | Core observables; directly compared to Kuczera 2021 literature values for ALA8 and to Shalongo-Stellwagen NMR data | Mean-first-passage-time (MFPT): k = 1/⟨τ⟩ |
| Relaxation time τ_eq | Single timescale of the helix-coil equilibrium; target for enhanced sampling validation | τ_eq = 1/(k_f + k_u) from counting; also from ACF fit |
| ACF of N_HB + biexponential fit | Independent, model-free estimate of τ_relax; biexponential because fast (sub-ns, local bond fluctuations) and slow (global folding) motions coexist | C(t) = A·exp(−t/τ_slow) + (1−A)·exp(−t/τ_fast) |
| ΔG_fold | Thermodynamic stability; directly from helix fraction and Boltzmann | ΔG = −RT ln(f_helix / (1−f_helix)) |

### Part 7 — A10 vs A8Q2 comparison + CV-transfer readiness
| What | Why |
|------|-----|
| Side-by-side 2D FES | The transferred TICA/SPIB model must reproduce this landscape for A8Q2; any shift in basin positions or relative depths indicates a failure of the transferred CV |
| Per-residue helix fraction overlay | Directly shows how the two Gln substitutions (positions 3, 8) change local helix propensity; this is the observable that validates or falsifies the transfer |
| Torsion covariance eigenvector overlap | Quantifies how similar the slow collective motions are between A10 and A8Q2; if PC-1 and PC-2 are nearly identical (overlap > 0.7), the TICA latent space of A10 is a good description of A8Q2 motion and the transfer is expected to succeed |

---

## Literature references

| Reference | What it provides |
|-----------|------------------|
| **Kuczera & Jas (2021)** *Life* 11, 385 (PMC8170890) | Complete kinetics protocol for ALA_n: H-bond threshold 3.6 Å; RMSD criterion; k_f/k_u values; ACF biexp fits; Table 2 is our primary benchmark |
| **Tavernelli et al. (2003)** *Biophys. J.* 85, 2641 | Justification for ΔG = −kBT ln P; minimum-referenced convention |
| **Kabsch & Sander (1983)** *Biopolymers* 22, 2577 | DSSP energy-based H-bond criterion (−0.5 kcal/mol threshold) |
| **Huang et al. (2017)** *Nat. Methods* 14, 71 | CHARMM36m validation; helix fraction of ALA5-based peptides |
| **Grossfield & Zuckerman (2009)** *Annu. Rep. Comput. Chem.* 5, 23 | Block Standard Error as the convergence metric; plateau criterion |
| **Best & Hummer (2009)** *J. Phys. Chem. B* 113, 9004 | Per-residue helix fraction definition from φ/ψ torsion angles |
| **Pérez-Hernández et al. (2013)** *J. Chem. Phys.* 139, 015102 | Sin/cos torsion features for TICA; covariance eigendecomposition |
| **MDAnalysis DSSP** v2.8+ | Implementation: https://docs.mdanalysis.org/stable/documentation_pages/analysis/dssp.html |

In [ ]:
"""
CELL 1 — Imports, physical constants, and global parameters

WHY this cell exists
--------------------
All physical constants and analysis thresholds are defined here in one place
so that a reader can change the temperature, threshold, or bin count once
and have every downstream cell automatically update.  No magic numbers
are buried inside plotting or analysis functions.

Dependency strategy
-------------------
MDTraj  : trajectory loading, RMSD, dihedral angles, distance computation
          Best-in-class for large DCD files; vectorised Cython backend.
MDAnalysis : DSSP (Kabsch-Sander) — MDTraj has no DSSP implementation;
             MDAnalysis v2.8 added the full algorithm.
Both libraries are optional-with-graceful-fallback so the notebook
still runs (partially) if one is missing.
"""
import os
import warnings
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec
from scipy.signal import correlate
from scipy.optimize import curve_fit

warnings.filterwarnings('ignore')
matplotlib.rcParams.update({
    'font.size': 12,
    'axes.titlesize': 13,
    'axes.labelsize': 12,
    'legend.fontsize': 10,
    'figure.dpi': 120,
})

# ── MDTraj ────────────────────────────────────────────────────────────────────
try:
    import mdtraj as md
    MDTRAJ_OK = True
    print(f'MDTraj     : {md.__version__}')
except ImportError:
    MDTRAJ_OK = False
    print('WARNING: MDTraj not found  →  pip install mdtraj')

# ── MDAnalysis ────────────────────────────────────────────────────────────────
# DSSP was added in 2.8.0; fall back gracefully if an older version is installed.
try:
    import MDAnalysis as mda
    MDA_OK = True
    print(f'MDAnalysis : {mda.__version__}')
    try:
        from MDAnalysis.analysis.dssp import DSSP
        MDA_DSSP_OK = True
    except ImportError:
        MDA_DSSP_OK = False
        print('  DSSP module not found (need >= 2.8.0) — DSSP criterion will use HB-frac proxy')
except ImportError:
    MDA_OK = MDA_DSSP_OK = False
    print('WARNING: MDAnalysis not found  →  pip install mdanalysis')

# ─── Physical constants ───────────────────────────────────────────────────────
# Boltzmann constant in the two unit systems commonly used in MD analysis.
kB_kcal  = 1.987204259e-3   # kcal mol⁻¹ K⁻¹
kB_kJ    = 8.314462618e-3   # kJ  mol⁻¹ K⁻¹

# 320 K is the simulation temperature chosen in the production notebook.
# At 300 K, helix relaxation time for ALA8 is ~12 ns (Kuczera 2021 Table 2).
# Raising to 320 K accelerates the folding/unfolding kinetics by ~5–8x via
# Arrhenius, giving ~2–3 ns relaxation and many more transitions per 1 µs run.
TEMP_K   = 320.0
kBT_kcal = kB_kcal * TEMP_K   # 0.6358 kcal/mol at 320 K
kBT_kJ   = kB_kJ   * TEMP_K   # 2.661  kJ/mol  at 320 K

# ─── Trajectory frame spacing ─────────────────────────────────────────────────
# DCD_INTERVAL_PS in the production notebook is 100 ps.
# A 1 µs run therefore has 10 000 frames.  All time-domain quantities (ACF lag,
# sojourn lengths, convergence plots) are computed in nanoseconds.
DT_PS  = 100.0
DT_NS  = DT_PS / 1000.0   # 0.1 ns per frame

# ─── FES display parameters ───────────────────────────────────────────────────
# 60 bins across the RMSD range (~0–6 Å) gives ~0.1 Å resolution — sufficient
# to resolve the helix and coil basins and the barrier between them.
# FES_CLIP: bins above 5 kcal/mol are populated by < exp(−5/0.636) ≈ 0.04% of
# frames, i.e. essentially empty; clipping avoids misleading colour stretching.
FES_BINS_1D = 60
FES_BINS_2D = 60
FES_CLIP    = 5.0   # kcal/mol

# ─── Two-state classification thresholds ─────────────────────────────────────
# RMSD (nm) — derived from Kuczera 2021 Table 1:
#   Helix cluster centroid RMSD ≈ 0.14 nm; coil centroid ≈ 0.28 nm.
#   We place the helix boundary at 0.20 nm (helix basin clearly below) and
#   the coil boundary at 0.30 nm (coil basin clearly above).
#   Frames between 0.20 and 0.30 nm are in the transition zone; they inherit
#   the previous state (hysteresis) to suppress flickering transitions.
RMSD_HELIX_NM = 0.20   # frames with RMSD < 0.20 nm  →  helix
RMSD_COIL_NM  = 0.30   # frames with RMSD > 0.30 nm  →  coil

# H-bond threshold: 50% of the maximum possible helical H-bonds (6 for 10-mer).
# Three or more H-bonds = central helix nucleus exists (Kuczera 2021 rationale).
HB_HELIX_FRAC   = 0.50

# DSSP threshold: same 50% rule applied to the per-residue helix fraction from DSSP.
DSSP_HELIX_FRAC = 0.50

# ─── Ideal α-helix Ramachandran centre ───────────────────────────────────────
# Standard α-helical values used to mark the ideal helix locus on Ramachandran
# plots.  Kuczera 2021 uses −62°/−41°; original Ramachandran (1963) gives −57°/−47°.
PHI_HELIX_DEG = -62.0
PSI_HELIX_DEG = -41.0

print()
print(f'Simulation temperature : {TEMP_K} K')
print(f'kBT                    : {kBT_kcal:.4f} kcal/mol  |  {kBT_kJ:.4f} kJ/mol')
print(f'Frame spacing          : {DT_PS} ps = {DT_NS:.3f} ns per frame')
print(f'Expected frames (1 µs) : {int(1000.0 / DT_NS)}')
print()
print('Two-state thresholds:')
print(f'  RMSD helix  < {RMSD_HELIX_NM*10:.0f} Å     coil  > {RMSD_COIL_NM*10:.0f} Å')
print(f'  HB frac    >= {HB_HELIX_FRAC:.0%}            DSSP frac >= {DSSP_HELIX_FRAC:.0%}')

In [ ]:
"""
CELL 2 — System definitions

WHY this structure
------------------
We define each (peptide × starting conformation) pair as a dictionary entry.
Every downstream cell loops over `available` — the subset with all files
present — so partial runs (e.g., A10 done but A8Q2 not yet) produce partial
but valid output rather than crashing.

File naming convention (set by production notebooks)
------------------------------------------------------
  {PREFIX}_{conf}_npt_final.pdb  — solvated topology; needed by MDAnalysis and
                                    MDTraj as the topology file for the DCD.
  {PREFIX}_{conf}_prod.dcd       — 1 µs NVT production trajectory (10 000 frames
                                    at 100 ps / frame).
  {PREFIX}_helix.pdb             — ideal capped helix structure in vacuum (no water),
                                    used as the RMSD reference.  The same reference
                                    is used for both helix and extended trajectories
                                    of the same peptide so RMSD values are directly
                                    comparable.

max_hb: maximum backbone α-helical H-bonds
-------------------------------------------
For Ac-Xaa₁₀-NHMe (12-residue capped chain: ACE + 10 AA + NME):
  H-bond pairs are O_i ··· HN_{i+4} for i = 0..6 (residue index 0 = ACE):
    ACE(0)-O ··· ALA(4)-HN
    ALA(1)-O ··· ALA(5)-HN
    ALA(2)-O ··· ALA(6)-HN
    ALA(3)-O ··· ALA(7)-HN
    ALA(4)-O ··· ALA(8)-HN
    ALA(5)-O ··· ALA(9)-HN
    ALA(6)-O ··· NME(10)-HN    ← NME methyl-amide has exactly one HN
  Total: 7 pairs.  However Kuczera 2021 counts only the 6 central pairs
  (excluding the ACE-CO...ALA4-HN pair, which is a cap-to-chain contact).
  We follow Kuczera's convention: max_hb = 6.
"""

SYSTEMS = [
    {
        'prefix'   : 'A10',
        'label'    : 'Ac-Ala₁₀-NHMe (helix start)',
        'conf'     : 'helix',
        'ref_pdb'  : 'A10_helix.pdb',
        'top_pdb'  : 'A10_helix_npt_final.pdb',
        'traj_dcd' : 'A10_helix_prod.dcd',
        'n_res'    : 10,
        'max_hb'   : 6,
        'color'    : '#e63946',
    },
    {
        'prefix'   : 'A10',
        'label'    : 'Ac-Ala₁₀-NHMe (extended start)',
        'conf'     : 'extended',
        'ref_pdb'  : 'A10_helix.pdb',          # same reference as helix-start
        'top_pdb'  : 'A10_extended_npt_final.pdb',
        'traj_dcd' : 'A10_extended_prod.dcd',
        'n_res'    : 10,
        'max_hb'   : 6,
        'color'    : '#457b9d',
    },
    {
        'prefix'   : 'A8Q2',
        'label'    : 'Ac-Ala₈Gln₂-NHMe (helix start)',
        'conf'     : 'helix',
        'ref_pdb'  : 'A8Q2_helix.pdb',
        'top_pdb'  : 'A8Q2_helix_npt_final.pdb',
        'traj_dcd' : 'A8Q2_helix_prod.dcd',
        'n_res'    : 10,
        'max_hb'   : 6,
        'color'    : '#f4a261',
    },
    {
        'prefix'   : 'A8Q2',
        'label'    : 'Ac-Ala₈Gln₂-NHMe (extended start)',
        'conf'     : 'extended',
        'ref_pdb'  : 'A8Q2_helix.pdb',          # same reference as helix-start
        'top_pdb'  : 'A8Q2_extended_npt_final.pdb',
        'traj_dcd' : 'A8Q2_extended_prod.dcd',
        'n_res'    : 10,
        'max_hb'   : 6,
        'color'    : '#2a9d8f',
    },
]

available = []
for s in SYSTEMS:
    needed  = [s['top_pdb'], s['traj_dcd'], s['ref_pdb']]
    missing = [f for f in needed if not os.path.exists(f)]
    if missing:
        print(f"[SKIP] {s['label']}")
        for m in missing:
            print(f"       missing: {m}")
    else:
        available.append(s)
        print(f"[OK]   {s['label']}")

print(f'\n{len(available)} / {len(SYSTEMS)} systems ready for analysis.')

In [ ]:
"""
CELL 3 — All helper / analysis functions

Organised into sections 3A–3H.  Each function has a detailed docstring
explaining the physical meaning, the specific method chosen, and any
caveats.  No analysis code is written inline in plotting cells.
"""

# ═════════════════════════════════════════════════════════════════════════════
# 3A  FREE ENERGY SURFACE
#
# WHY Boltzmann inversion?
#   MD generates frames proportional to the Boltzmann weight e^{-U/kBT}.
#   The histogram P(x) is therefore proportional to exp(−ΔG(x)/kBT).
#   Inverting: ΔG(x) = −kBT ln P(x).  We normalise by P_max so the
#   global minimum is at 0 kcal/mol — a universal convention that makes
#   FES comparisons between different systems straightforward.
#
# WHY histogram-based and not KDE?
#   Histograms are deterministic and reproducible: the same input always
#   produces the same output regardless of bandwidth selection.  KDE
#   introduces a smoothing parameter that the analyst must justify and
#   that can smear out real barriers.  For reporting ground-truth FES we
#   prefer the conservative choice (Tavernelli 2003).
# ═════════════════════════════════════════════════════════════════════════════

def fes_1d(x, bins=FES_BINS_1D, kBT=kBT_kcal):
    """
    1D free energy surface via Boltzmann inversion of a histogram.

    ΔG(x) = −kBT · ln[P(x) / P_max]

    The minimum is set to 0 (P_max bin is the reference state).
    Unpopulated bins (count = 0) return np.nan so contourf skips them
    rather than showing a misleadingly high energy value.

    Parameters
    ----------
    x    : 1D array of the order parameter values
    bins : number of histogram bins
    kBT  : thermal energy in desired output units

    Returns
    -------
    centres : bin centre values
    G       : ΔG in kcal/mol (or kJ/mol if kBT is in kJ/mol)
    """
    counts, edges = np.histogram(x, bins=bins, density=False)
    centres = 0.5 * (edges[:-1] + edges[1:])
    c = counts.astype(float)
    with np.errstate(divide='ignore', invalid='ignore'):
        G = np.where(c > 0, -kBT * np.log(c / c.max()), np.nan)
    return centres, G


def fes_2d(x, y, bins=FES_BINS_2D, kBT=kBT_kcal, clip=FES_CLIP):
    """
    2D free energy surface via Boltzmann inversion of a 2D histogram.

    ΔG(x, y) = −kBT · ln[P(x, y) / P_max]

    The `clip` parameter sets the maximum displayed free energy.  Bins
    above the clip value have negligible population and their exact ΔG
    is unreliable; clipping keeps the colour scale focused on the
    physically meaningful region (basins and barriers).

    Returns meshgrid arrays (xi, yi, G) suitable for contourf.
    Both x and y must be float arrays for numpy histogram2d to produce
    correct floating-point bin edges.
    """
    H, xe, ye = np.histogram2d(x, y, bins=bins, density=False)
    xc = 0.5 * (xe[:-1] + xe[1:])
    yc = 0.5 * (ye[:-1] + ye[1:])
    xi, yi = np.meshgrid(xc, yc, indexing='ij')
    H = H.astype(float)
    with np.errstate(divide='ignore', invalid='ignore'):
        G = np.where(H > 0, -kBT * np.log(H / H.max()), np.nan)
    G = np.clip(G, 0.0, clip)
    return xi, yi, G


# ═════════════════════════════════════════════════════════════════════════════
# 3B  TRAJECTORY LOADING AND ALIGNMENT  (MDTraj)
#
# WHY strip solvent BEFORE aligning?
#   md.load() returns a trajectory that includes all atoms: peptide, water,
#   ions.  MDTraj.superpose and md.rmsd operate on all atoms if atom_indices
#   is not explicitly given (and the default is 'all').  If water molecules
#   are present during the Cα superpose step, the optimised rotation matrix
#   will be dominated by the far more numerous water atoms, making the
#   alignment meaningless.  Slicing to protein-only atoms first is mandatory.
#
# WHY use precentered=True in md.rmsd?
#   After superpose(), each frame is already in the reference frame.
#   md.rmsd() by default re-superposes internally.  Using precentered=True
#   skips the internal superpose so we compute RMSD from the already-aligned
#   coordinates rather than doing a second, potentially different, alignment.
# ═════════════════════════════════════════════════════════════════════════════

def load_trajectory(top_pdb, traj_dcd, ref_pdb):
    """
    Load a solvated DCD trajectory, strip solvent, superpose each frame on
    the Cα atoms of the ideal helix reference structure, and return the
    stripped, aligned trajectory together with the stripped reference.

    The reference PDB must be the capped, vacuum helix structure (no water)
    that was written by Phase 1 of the production notebook.  Using a vacuum
    reference ensures the RMSD measures structural deviation from the ideal
    helix conformation rather than from a snapshot that happened to be in
    a particular solvent configuration.

    Returns
    -------
    traj_prot : MDTraj trajectory, peptide-only, superposed on ref Cα
    ref_prot  : MDTraj single-frame trajectory of the reference structure
    """
    traj = md.load(traj_dcd, top=top_pdb)
    ref  = md.load(ref_pdb)

    # Include ACE and NME in the 'protein' slice because MDTraj's 'protein'
    # selection excludes capping groups by default.
    prot_sel     = traj.topology.select('protein or resname ACE or resname NME')
    traj_prot    = traj.atom_slice(prot_sel)

    ref_prot_sel = ref.topology.select('protein or resname ACE or resname NME')
    ref_prot     = ref.atom_slice(ref_prot_sel)

    # Build matching Cα index arrays; truncate to the shorter chain if lengths
    # differ (should not happen for same-sequence systems).
    ca_traj = traj_prot.topology.select('name CA')
    ca_ref  = ref_prot.topology.select('name CA')
    n_ca    = min(len(ca_traj), len(ca_ref))

    # superpose modifies traj_prot in-place: all frames are now in the frame
    # of the reference structure, rotated/translated to minimise Cα RMSD.
    traj_prot.superpose(ref_prot,
                        atom_indices=ca_traj[:n_ca],
                        ref_atom_indices=ca_ref[:n_ca])
    return traj_prot, ref_prot


def compute_rmsd(traj_prot, ref_prot):
    """
    Per-frame backbone (N, Cα, C, O) RMSD to the reference helix, in nm.

    WHY backbone heavy atoms instead of Cα only?
      Cα-only RMSD can miss helices with distorted bond angles or sheet-like
      packing of adjacent turns.  Including N, C, O atoms captures backbone
      geometry changes more completely and is the standard in Kuczera 2021.

    WHY precentered=True?
      The trajectory was superposed in load_trajectory().  Re-superposes are
      redundant and may produce slightly different rotation matrices due to
      floating-point differences, introducing noise into the RMSD series.

    Returns rmsd_nm : (n_frames,) float array, nanometres.
    """
    bb_traj = traj_prot.topology.select('backbone')
    bb_ref  = ref_prot.topology.select('backbone')
    n_bb    = min(len(bb_traj), len(bb_ref))
    return md.rmsd(
        traj_prot, ref_prot,
        atom_indices=bb_traj[:n_bb],
        ref_atom_indices=bb_ref[:n_bb],
        precentered=True,
    )


def compute_phi_psi(traj_prot):
    """
    Backbone φ and ψ dihedral angles for all standard (non-cap) residues.

    MDTraj.compute_phi / compute_psi automatically skip ACE (no preceding
    residue for φ) and NME (no following residue for ψ), returning arrays
    of shape (n_frames, n_internal_residues) in radians.

    WHY radians?
      Radians are the native output of MDTraj and are required for sin/cos
      encoding in the covariance analysis (Cell 13).  Conversion to degrees
      is done locally in display cells.

    Returns (phi, psi) in radians, or (None, None) on failure.
    """
    try:
        _, phi = md.compute_phi(traj_prot)
        _, psi = md.compute_psi(traj_prot)
        return phi, psi
    except Exception as e:
        print(f'  WARNING: phi/psi computation failed: {e}')
        return None, None


# ═════════════════════════════════════════════════════════════════════════════
# 3C  HELICAL H-BOND COUNTING  (MDTraj)
#
# WHY a distance-only criterion (no angle)?
#   Kuczera 2021 and Baker-Hubbard 1984 both use a pure donor-acceptor
#   distance threshold.  For backbone α-helical H-bonds the NH···O angle
#   is geometrically constrained to ~160° simply by the helical geometry,
#   so a distance threshold of 3.6 Å already implicitly enforces a reasonable
#   angle.  Adding an explicit angle threshold would only ever exclude frames
#   at the very boundary of the helix basin, changing f_helix by < 1%.
#
# WHY 3.6 Å (0.36 nm)?
#   The α-helical equilibrium H···O distance is ~2.0 Å; with the covalent
#   N-H bond of 1.0 Å the N···O distance is ~3.0 Å.  The standard cutoff
#   of 3.5–3.6 Å allows for thermal fluctuations while excluding contacts
#   that are clearly not H-bonds.  Kuczera 2021 explicitly uses 3.6 Å.
#
# WHY include ACE and NME?
#   ACE has a carbonyl O that can accept an H-bond from ALA(4)-HN, and NME
#   has an amide H (HN) that can donate to ALA(6)-O.  These are real helical
#   H-bonds present in the capped peptide and counted by Kuczera 2021.
#   Omitting them would systematically undercount N_HB by 1–2 bonds per
#   frame in the helix state.
#
# WHY bond-graph-based H detection?
#   CHARMM36m names the amide H as 'HN'.  PDBFixer / other preparators may
#   produce 'H'.  N-terminal residues have 'H1', 'H2', 'H3'.  A name-only
#   search for 'H' would miss all CHARMM-named donors.  Traversing the
#   bond graph (which atom is bonded to N?) is topology-agnostic.
# ═════════════════════════════════════════════════════════════════════════════

def count_helix_hbonds(traj_prot, max_hb=6):
    """
    Count backbone α-helical H-bonds per frame and return per-bond occupancies.

    H-bond criterion: H(i+4) ··· O(i) distance < 3.6 Å (0.36 nm),
    for all i-(i+4) backbone pairs in the topology.

    Returns
    -------
    hb_count     : (n_frames,) int array — total H-bonds per frame, clipped to max_hb
    hb_occupancy : (n_pairs,) float array — fraction of frames each bond is formed
                   (n_pairs ≤ max_hb; useful for per-bond occupancy bar chart)
    n_pairs      : actual number of i-(i+4) pairs found in the topology
    """
    top     = traj_prot.topology
    all_res = list(top.residues)

    # Per-residue: find acceptor O (carbonyl) and donor H (amide) atom indices.
    acceptor_O = []   # carbonyl O index, or None
    donor_H    = []   # amide H  index, or None

    # Names that can be the amide hydrogen across different force-field/preparator conventions
    H_NAMES = {'H', 'HN', 'H1', 'HT1', 'HT2', 'HT3', 'HN1'}

    for res in all_res:
        # Carbonyl O: always named 'O' in backbone
        o_list = [a.index for a in res.atoms if a.name == 'O']

        # Amide H: search by name AND verify it is bonded to a nitrogen.
        # This correctly handles CHARMM ('HN'), AMBER ('H'), and terminal groups.
        h_list = [
            a.index for a in res.atoms
            if a.name in H_NAMES and
            any(
                b.atom1.name == 'N' or b.atom2.name == 'N'
                for b in top.bonds
                if a.index in (b.atom1.index, b.atom2.index)
            )
        ]

        # Fallback: find any H directly bonded to backbone N (catches edge cases)
        if not h_list:
            for a in res.atoms:
                if a.name == 'N':
                    bonded = [
                        (b.atom2.index if b.atom1.index == a.index else b.atom1.index)
                        for b in top.bonds
                        if a.index in (b.atom1.index, b.atom2.index)
                    ]
                    h_list = [i for i in bonded if top.atom(i).element.symbol == 'H']
                    if h_list:
                        break

        acceptor_O.append(o_list[0] if o_list else None)
        donor_H.append(h_list[0]    if h_list else None)

    # Identify i-(i+4) (H, O) pairs
    n = len(all_res)
    hb_pairs = []
    pair_labels = []   # e.g. 'ACE(0)→ALA(4)' for labelling the occupancy plot
    for i in range(n - 4):
        acc = acceptor_O[i]
        don = donor_H[i + 4]
        if acc is not None and don is not None:
            hb_pairs.append((don, acc))
            pair_labels.append(f'{all_res[i].name}({i})→{all_res[i+4].name}({i+4})')

    if not hb_pairs:
        print('  WARNING: No i-(i+4) H-bond pairs found. Returning zeros.')
        zeros = np.zeros(traj_prot.n_frames, dtype=int)
        return zeros, np.array([]), 0

    # Compute all H···O distances in a single vectorised call: (n_frames, n_pairs) nm
    dists    = md.compute_distances(traj_prot, np.array(hb_pairs))   # nm
    formed   = (dists < 0.36).astype(int)   # bool array: 1 = bond formed

    hb_count     = np.clip(formed.sum(axis=1), 0, max_hb)
    hb_occupancy = formed.mean(axis=0)   # fraction of frames each bond is formed

    return hb_count, hb_occupancy, len(hb_pairs)


# ═════════════════════════════════════════════════════════════════════════════
# 3D  DSSP VIA MDAnalysis
#
# WHY MDAnalysis for DSSP instead of MDTraj?
#   MDTraj does not implement DSSP.  MDAnalysis v2.8+ includes the full
#   Kabsch-Sander (1983) energy-based algorithm which computes an electrostatic
#   H-bond energy E = 0.084 × [1/r_ON + 1/r_CH − 1/r_OH − 1/r_CN] (kcal/mol)
#   and assigns 'H' if E < −0.5 kcal/mol.  This is distinct from the simple
#   distance criterion in 3C and provides an independent assessment.
#
# WHY does DSSP return strings ('H', 'E', 'C', ...) and not one-hot arrays?
#   MDAnalysis DSSP (v2.8) returns `results.dssp` as a 2D object array of
#   single-character strings with shape (n_frames, n_residues).  The correct
#   way to extract helix is `(dssp_codes == 'H')`, which broadcasts to a
#   boolean array.  There is no 3-channel one-hot encoding — that was a bug
#   in the original version of this notebook.
# ═════════════════════════════════════════════════════════════════════════════

def compute_dssp_helix_frac(top_pdb, traj_dcd):
    """
    Per-frame and per-residue α-helix fraction from MDAnalysis DSSP.

    Returns
    -------
    per_frame   : (n_frames,) float — fraction of residues assigned 'H' per frame
    per_residue : (n_residues,) float — fraction of frames each residue is 'H'
    """
    u       = mda.Universe(top_pdb, traj_dcd)
    protein = u.select_atoms('protein')
    run     = DSSP(protein).run()

    dssp_codes  = run.results.dssp           # (n_frames, n_residues), dtype=object
    is_helix    = (dssp_codes == 'H')        # bool (n_frames, n_residues)
    per_frame   = is_helix.mean(axis=1)      # (n_frames,)
    per_residue = is_helix.mean(axis=0)      # (n_residues,)
    return per_frame, per_residue


# ═════════════════════════════════════════════════════════════════════════════
# 3E  TWO-STATE CLASSIFICATION
#
# WHY three independent criteria?
#   Any single criterion can be gamed by choosing a convenient threshold.
#   Three physically distinct criteria (shape/RMSD, electrostatic H-bond count,
#   algorithm-based DSSP) are unlikely to agree spuriously.  If all three assign
#   the same state to > 90% of frames, the state assignment is trustworthy.
#
# WHY hysteresis on the RMSD criterion only?
#   RMSD is a continuous scalar and frames can oscillate between basins rapidly.
#   H-bond count is a discrete integer; changing by one bond is a distinct event,
#   not noise.  DSSP is energy-based and has its own implicit smoothing.  Only the
#   continuous RMSD criterion benefits from hysteresis to suppress artefactual
#   rapid-fire transitions.
# ═════════════════════════════════════════════════════════════════════════════

def classify_rmsd_hysteresis(rmsd_nm,
                              helix_max=RMSD_HELIX_NM,
                              coil_min=RMSD_COIL_NM):
    """
    Two-state (helix=True / coil=False) classification from RMSD with hysteresis.

    Rules:
      RMSD < helix_max          →  helix  (clear helix basin)
      RMSD > coil_min           →  coil   (clear coil  basin)
      helix_max ≤ RMSD ≤ coil_min → inherit state of previous frame (hysteresis zone)

    First frame: classified at midpoint (helix if below midpoint).
    Returns bool array (n_frames,), True = helix.
    """
    n   = len(rmsd_nm)
    out = np.zeros(n, dtype=bool)
    mid = 0.5 * (helix_max + coil_min)
    out[0] = (rmsd_nm[0] < mid)
    for i in range(1, n):
        r = rmsd_nm[i]
        if   r < helix_max:  out[i] = True
        elif r > coil_min:   out[i] = False
        else:                out[i] = out[i - 1]   # hysteresis
    return out


def classify_hbond(hb_count, max_hb, frac=HB_HELIX_FRAC):
    """
    Classify each frame as helix (True) if hb_count / max_hb >= frac.
    No hysteresis: the H-bond count is a discrete integer and the transition
    from 2 to 3 bonds is a genuine event, not continuous-variable noise.
    """
    return (hb_count / max_hb) >= frac


def classify_dssp(helix_frac, frac=DSSP_HELIX_FRAC):
    """
    Classify each frame as helix (True) if DSSP helix fraction >= frac.
    No hysteresis: DSSP already embeds its own energy-based stability criterion.
    """
    return helix_frac >= frac


# ═════════════════════════════════════════════════════════════════════════════
# 3F  TRANSITION COUNTING AND KINETICS
#
# WHY mean-first-passage-time (MFPT)?
#   MFPT is the simplest consistent estimator for two-state rate constants:
#   k = 1 / <τ_sojourn>.  It is exact for a Markovian two-state system and
#   is the method used in Kuczera 2021.  More elaborate methods (survival
#   function fitting, max-likelihood exponential) converge to the same answer
#   when N_transitions is large enough, but require stronger distributional
#   assumptions.
#
# WHY count complete sojourns only?
#   The first and last sojourn in a trajectory may be truncated (the
#   simulation starts or ends mid-state).  Including truncated sojourns
#   biases the mean downward (they appear shorter than they are).  Only
#   sojourns with both an entry event and an exit event are counted.
#
# Rate constant sign convention
#   k_u = 1 / <τ_H> : unfolding rate (helix → coil)
#   k_f = 1 / <τ_C> : folding   rate (coil  → helix)
#   τ_eq = 1 / (k_f + k_u) : global relaxation time; this is what an
#           ACF fit measures and what enhanced sampling aims to accelerate.
# ═════════════════════════════════════════════════════════════════════════════

def count_transitions(state, dt_ns=DT_NS):
    """
    Kinetic observables from a binary state time series (True = helix).

    Parameters
    ----------
    state : (n_frames,) bool array
    dt_ns : frame spacing in nanoseconds

    Returns
    -------
    dict with keys:
      n_fold, n_unfold    : number of C→H and H→C transitions
      tau_H_ns, tau_C_ns  : mean helix/coil residence times (ns)
      k_f_ns, k_u_ns      : folding/unfolding rate constants (ns⁻¹)
      tau_eq_ns           : global relaxation time = 1/(k_f + k_u)
      f_helix             : fraction of frames in helix state
      dG_kcal             : folding free energy (kcal/mol)
      h_sojourns          : helix residence time of each individual sojourn (ns)
      c_sojourns          : coil  residence time of each individual sojourn (ns)
      fold_frames         : frame indices where helix periods begin (C→H)
      unfold_frames       : frame indices where coil  periods begin (H→C)
    """
    s = state.astype(int)
    d = np.diff(s)                           # +1: coil→helix, −1: helix→coil
    fold_frames   = np.where(d ==  1)[0] + 1  # first helix frame
    unfold_frames = np.where(d == -1)[0] + 1  # first coil  frame

    def sojourns(enter_idx, exit_idx):
        """Lengths (ns) of complete sojourns: enter event followed by exit event."""
        lengths = []
        for e in enter_idx:
            after = exit_idx[exit_idx > e]
            if len(after):
                lengths.append((after[0] - e) * dt_ns)
        return np.array(lengths) if lengths else np.array([0.0])

    h_soj = sojourns(fold_frames,   unfold_frames)
    c_soj = sojourns(unfold_frames, fold_frames)

    tau_H  = h_soj.mean() if len(h_soj) else np.nan
    tau_C  = c_soj.mean() if len(c_soj) else np.nan

    # Guard against tau = 0 (physically impossible) and NaN
    k_u = 1.0 / tau_H if (tau_H > 0 and not np.isnan(tau_H)) else np.nan
    k_f = 1.0 / tau_C if (tau_C > 0 and not np.isnan(tau_C)) else np.nan

    tau_eq = (1.0 / (k_f + k_u)
              if not (np.isnan(k_f) or np.isnan(k_u)) else np.nan)

    f_helix = s.mean()
    dG_kcal = (
        -kBT_kcal * np.log(f_helix / (1.0 - f_helix))
        if 0.0 < f_helix < 1.0 else np.nan
    )

    return {
        'n_fold'        : len(fold_frames),
        'n_unfold'      : len(unfold_frames),
        'tau_H_ns'      : tau_H,
        'tau_C_ns'      : tau_C,
        'k_f_ns'        : k_f,
        'k_u_ns'        : k_u,
        'tau_eq_ns'     : tau_eq,
        'f_helix'       : f_helix,
        'dG_kcal'       : dG_kcal,
        'h_sojourns'    : h_soj,
        'c_sojourns'    : c_soj,
        'fold_frames'   : fold_frames,
        'unfold_frames' : unfold_frames,
    }


# ═════════════════════════════════════════════════════════════════════════════
# 3G  AUTOCORRELATION FUNCTION
#
# WHY ACF of N_HB (not RMSD)?
#   N_HB is a near-integer order parameter that strongly separates the two
#   states.  Its ACF decays to zero with a characteristic timescale τ_relax
#   that equals τ_eq = 1/(k_f + k_u) for a two-state system.  RMSD varies
#   more within each state, mixing intra-state fluctuations into the ACF and
#   making the fit less clean.  Kuczera 2021 uses N_HB for the ACF.
#
# WHY biexponential fit?
#   Two relaxation processes coexist: (1) fast, local backbone fluctuations
#   (τ_fast ~ sub-ns to 1 ns) and (2) slow, global helix-coil transitions
#   (τ_slow ~ a few ns at 320 K).  A single exponential misses the fast
#   process and overestimates τ_slow.  A biexponential captures both and
#   τ_slow should agree with τ_eq from transition counting (cross-validation).
#
# WHY FFT-based (scipy.signal.correlate)?
#   For n = 10 000 frames a direct-sum ACF computation is O(n²) ≈ 10⁸
#   operations.  scipy.signal.correlate uses FFT which is O(n log n) ≈ 10⁵,
#   making it ~1000x faster.
# ═════════════════════════════════════════════════════════════════════════════

def compute_acf(x, max_lag_ns=200.0, dt_ns=DT_NS):
    """
    Normalised autocorrelation function of x, up to max_lag_ns.

    Normalisation: ACF(0) = 1 by construction (divide by variance * N).
    If var(x) == 0 (constant signal), returns a flat ACF of 1 throughout.

    Returns (lag_ns, acf) arrays.
    """
    x_c = x - x.mean()
    var = x_c.var()
    if var == 0:
        n_lag = int(max_lag_ns / dt_ns) + 1
        return np.arange(n_lag) * dt_ns, np.ones(n_lag)

    n_lag    = min(len(x), int(max_lag_ns / dt_ns) + 1)
    full     = correlate(x_c, x_c, mode='full')
    pos      = full[len(x) - 1 : len(x) - 1 + n_lag]
    acf_norm = pos / (var * len(x))
    lag_ns   = np.arange(n_lag) * dt_ns
    return lag_ns, acf_norm


def fit_biexponential(t, acf):
    """
    Fit ACF to  A·exp(−t/τ₁) + (1−A)·exp(−t/τ₂).

    WHY fit only where ACF > 0.02?
      In the tail (ACF near zero) the signal is dominated by noise from
      finite-trajectory averaging.  Fitting the positive lobe only avoids
      letting noisy tail fluctuations degrade the estimate of τ_slow.

    WHY bounds (0, t_max * 10) for the timescales?
      Lower bound 1e-6 ns prevents division-by-zero.  Upper bound t_max * 10
      allows τ_slow to be much larger than the fit window (poorly sampled
      transitions will give a large τ_slow) while still constraining the
      optimiser from wandering to infinity.

    Returns (popt, tau_slow, tau_fast, A_amplitude) or None on failure.
    """
    def biexp(t, A, tau1, tau2):
        return A * np.exp(-t / tau1) + (1.0 - A) * np.exp(-t / tau2)

    mask  = acf > 0.02
    t_fit = t[mask]
    y_fit = acf[mask]
    if len(t_fit) < 5:
        return None

    t_max = t_fit[-1]
    try:
        popt, _ = curve_fit(
            biexp, t_fit, y_fit,
            p0     = [0.7, t_max / 2.0, t_max / 10.0],
            bounds = ([0.0, 1e-6, 1e-6], [1.0, t_max * 10, t_max * 10]),
            maxfev = 100_000,
        )
        A, tau1, tau2 = popt
        return popt, max(tau1, tau2), min(tau1, tau2), A
    except RuntimeError:
        return None


# ═════════════════════════════════════════════════════════════════════════════
# 3H  BLOCK STANDARD ERROR  (BSE)
#
# WHY BSE instead of a simple SEM?
#   Standard error of the mean (SEM = std / sqrt(N)) assumes independent
#   samples.  In an MD trajectory consecutive frames are highly correlated
#   (autocorrelation time ~ 2 ns; frame spacing = 0.1 ns), so N_independent
#   << N_frames.  Using N_frames gives a SEM that is ~4x too small.
#
#   BSE groups frames into blocks of length b, computes the mean of each
#   block, and then computes the standard error across blocks.  When b >>
#   τ_correlation, block means are approximately independent and the BSE
#   converges to the true uncertainty.  Plotting BSE vs b and finding the
#   plateau is the standard protocol (Grossfield & Zuckerman 2009).
#
# WHY log-spaced block sizes?
#   We need to cover a range from 1 frame (0.1 ns) to 25% of the trajectory
#   (250 ns).  Linearly spaced would waste resolution at small b where the
#   BSE rises steeply.  Log spacing gives equal coverage per decade.
# ═════════════════════════════════════════════════════════════════════════════

def block_standard_error(x, min_blocks=4, max_block_frac=0.25):
    """
    Block Standard Error (BSE) of the mean of x as a function of block size.

    For each block size b (in frames):
      1. Divide x into floor(N/b) non-overlapping blocks.
      2. Compute the mean of each block.
      3. BSE = std(block_means, ddof=1) / sqrt(n_blocks).

    Returns
    -------
    block_ns  : block sizes in nanoseconds (log-spaced)
    bse_vals  : corresponding BSE values
    """
    n     = len(x)
    max_b = max(1, int(n * max_block_frac))
    b_arr = np.unique(np.logspace(0, np.log10(max_b), num=30).astype(int))
    b_arr = b_arr[b_arr >= min_blocks]

    block_ns = []
    bse_vals = []
    for b in b_arr:
        n_blocks = n // b
        if n_blocks < 2:
            continue
        blocks = x[:n_blocks * b].reshape(n_blocks, b)
        bm     = blocks.mean(axis=1)
        bse    = bm.std(ddof=1) / np.sqrt(n_blocks)
        block_ns.append(b * DT_NS)
        bse_vals.append(bse)

    return np.array(block_ns), np.array(bse_vals)


# ═════════════════════════════════════════════════════════════════════════════
# 3I  PER-RESIDUE HELIX FRACTION FROM TORSION ANGLES
#
# WHY torsion-based instead of DSSP?
#   Torsion-based helix fraction is the most widely used definition in
#   CHARMM36m validation papers (Best & Hummer 2009; Huang 2017).  Each
#   residue is independently assessed by whether its (φ, ψ) falls in the
#   α-helical basin.  This allows plotting a per-residue 'helicity profile'
#   that directly maps onto experimental NMR chemical shift deviations and
#   CD spectra reconstructed from per-residue contributions.
#
# Basin definition: φ ∈ (−100°, −20°) and ψ ∈ (−80°, 0°)
#   This is the 'α' region of the Ramachandran plot as defined by
#   Best & Hummer 2009 and is the standard used in CHARMM36m validation.
#   It is broader than the Kuczera 20° circle (±20° from ideal) and therefore
#   slightly more permissive, but both give very similar f_helix values.
# ═════════════════════════════════════════════════════════════════════════════

def per_residue_helix_frac_torsion(phi, psi):
    """
    Per-residue helix fraction from backbone torsion angles.

    A residue is in the α-helical state if:
      φ ∈ (−100°, −20°)  AND  ψ ∈ (−80°, 0°)
    (Best & Hummer 2009 definition)

    Parameters
    ----------
    phi, psi : (n_frames, n_res) float arrays in radians

    Returns
    -------
    helix_frac : (n_res,) float array — fraction of frames in α-helical basin
    """
    phi_d = np.degrees(phi)
    psi_d = np.degrees(psi)
    in_helix = (
        (phi_d > -100.0) & (phi_d < -20.0) &
        (psi_d >  -80.0) & (psi_d <   0.0)
    )
    return in_helix.mean(axis=0)   # average over frames for each residue


print('All helper functions defined successfully.')

In [ ]:
"""
CELL 4 — Load trajectories and compute all order parameters

WHY one pass?
-------------
DCD trajectories can be large (several GB for 1 µs at 100 ps/frame).
We load each trajectory exactly once, compute every required observable
in memory, and store only the compact derived arrays in `results`.
The full traj object is NOT stored (it would exhaust RAM for four systems
simultaneously); instead, we store the per-bond H-bond formation array
so downstream cells have access to per-bond occupancies without reloading.

WHY store `hb_occupancy` separately?
-------------------------------------
Cell 8 needs per-bond occupancy fractions (one value per i→i+4 pair)
for the bar chart.  `hb_count` only gives the total per frame.
The function `count_helix_hbonds` now returns both.
"""

results = {}

for s in available:
    key = (s['prefix'], s['conf'])
    print(f"\n{'='*64}")
    print(f"  {s['label']}")
    print(f"{'='*64}")

    if not MDTRAJ_OK:
        print('  MDTraj unavailable — cannot proceed.  pip install mdtraj')
        continue

    # ── 1. Load, strip solvent, align on Cα of ideal helix ───────────────────
    print('  Loading & aligning...')
    traj_prot, ref_prot = load_trajectory(s['top_pdb'], s['traj_dcd'], s['ref_pdb'])
    n_frames = traj_prot.n_frames
    time_ns  = np.arange(n_frames) * DT_NS
    print(f'  Frames : {n_frames}  ({n_frames * DT_NS / 1000:.3f} µs)')

    # ── 2. Backbone RMSD to ideal α-helix ────────────────────────────────────
    # Used as: primary two-state reaction coordinate, 2D FES x-axis, convergence check
    print('  RMSD...')
    rmsd_nm = compute_rmsd(traj_prot, ref_prot)
    print(f'  RMSD  min={rmsd_nm.min()*10:.2f} Å  mean={rmsd_nm.mean()*10:.2f} Å  max={rmsd_nm.max()*10:.2f} Å')

    # ── 3. Backbone φ/ψ torsions ──────────────────────────────────────────────
    # Used as: Ramachandran PMF, per-residue helix fraction, CV covariance features
    print('  φ/ψ torsions...')
    phi, psi = compute_phi_psi(traj_prot)

    # ── 4. α-helical H-bond count (per frame and per bond) ───────────────────
    # Used as: 2D FES y-axis, 1D FES, H-bond-criterion state classification,
    #          ACF order parameter, per-bond occupancy bar chart
    print('  Helical H-bonds...')
    hb_count, hb_occupancy, n_pairs = count_helix_hbonds(traj_prot, max_hb=s['max_hb'])
    hb_frac = hb_count / s['max_hb']
    print(f'  H-bonds: {n_pairs} pairs  mean_count={hb_count.mean():.2f}  '
          f'mean_frac={hb_frac.mean():.3f}')

    # ── 5. DSSP helix fraction ────────────────────────────────────────────────
    # Used as: third independent state criterion, per-residue helix propensity
    if MDA_DSSP_OK:
        print('  DSSP (MDAnalysis)...')
        try:
            dssp_per_frame, dssp_per_res = compute_dssp_helix_frac(
                s['top_pdb'], s['traj_dcd']
            )
            print(f'  DSSP  mean={dssp_per_frame.mean():.3f}')
        except Exception as e:
            print(f'  DSSP failed ({e}) — using HB fraction as proxy')
            dssp_per_frame = hb_frac.copy()
            dssp_per_res   = None
    else:
        dssp_per_frame = hb_frac.copy()
        dssp_per_res   = None
        print('  DSSP skipped (need MDAnalysis >= 2.8.0) — using HB frac as proxy')

    # ── 6. Per-residue helix fraction from φ/ψ ────────────────────────────────
    # Used as: position-dependent helix propensity bar chart, A10 vs A8Q2 overlay
    per_res_frac = None
    if phi is not None:
        per_res_frac = per_residue_helix_frac_torsion(phi, psi)

    # ── 7. Three-criterion state assignment ───────────────────────────────────
    state_rmsd = classify_rmsd_hysteresis(rmsd_nm)
    state_hb   = classify_hbond(hb_count, s['max_hb'])
    state_dssp = classify_dssp(dssp_per_frame)

    # Agreement score: 0 = all criteria say coil, 3 = all say helix
    agree_score   = (state_rmsd.astype(int) +
                     state_hb.astype(int)   +
                     state_dssp.astype(int))

    # Primary state: RMSD-based (hysteresis applied; Kuczera 2021 standard)
    state_primary = state_rmsd

    agree_rmsd_hb = (state_rmsd == state_hb).mean()
    print(f'  State agreement RMSD vs HB : {agree_rmsd_hb:.1%}')

    # ── 8. Transition counting and kinetics ───────────────────────────────────
    print('  Transition counting...')
    kin = count_transitions(state_primary)
    print(f'  f_helix={kin["f_helix"]:.3f}  '
          f'ΔG={kin["dG_kcal"]:.2f} kcal/mol  '
          f'N_fold={kin["n_fold"]}  N_unfold={kin["n_unfold"]}')
    print(f'  τ_H={kin["tau_H_ns"]:.1f} ns  '
          f'τ_C={kin["tau_C_ns"]:.1f} ns  '
          f'τ_eq={kin["tau_eq_ns"]:.1f} ns')
    print(f'  k_f={kin["k_f_ns"]:.4f} ns⁻¹  k_u={kin["k_u_ns"]:.4f} ns⁻¹')

    # ── 9. ACF of H-bond order parameter ──────────────────────────────────────
    # Cross-validates τ_eq from counting; should agree within factor ~2
    print('  ACF...')
    acf_lag, acf_vals = compute_acf(hb_frac)
    acf_fit = fit_biexponential(acf_lag, acf_vals)
    if acf_fit is not None:
        _, tau_slow, tau_fast, A_amp = acf_fit
        print(f'  ACF biexp: τ_slow={tau_slow:.1f} ns  τ_fast={tau_fast:.2f} ns  A={A_amp:.2f}')
    else:
        tau_slow = tau_fast = A_amp = np.nan
        print('  ACF biexp fit failed (possibly too few transitions)')

    # ── 10. Block Standard Error ───────────────────────────────────────────────
    print('  BSE...')
    bse_bs, bse_vals = block_standard_error(state_primary.astype(float))

    # ── Store all derived data ────────────────────────────────────────────────
    results[key] = {
        'system'         : s,
        'n_frames'       : n_frames,
        'time_ns'        : time_ns,
        'rmsd_nm'        : rmsd_nm,
        'hb_count'       : hb_count,
        'hb_frac'        : hb_frac,
        'hb_occupancy'   : hb_occupancy,   # (n_pairs,) — fraction each bond is formed
        'n_hb_pairs'     : n_pairs,
        'dssp_per_frame' : dssp_per_frame,
        'dssp_per_res'   : dssp_per_res,
        'phi'            : phi,
        'psi'            : psi,
        'per_res_frac'   : per_res_frac,
        'state_rmsd'     : state_rmsd,
        'state_hb'       : state_hb,
        'state_dssp'     : state_dssp,
        'agree_score'    : agree_score,
        'state_primary'  : state_primary,
        'kinetics'       : kin,
        'acf_lag'        : acf_lag,
        'acf_vals'       : acf_vals,
        'acf_fit'        : acf_fit,
        'tau_slow_ns'    : tau_slow,
        'tau_fast_ns'    : tau_fast,
        'bse_bs'         : bse_bs,
        'bse_vals'       : bse_vals,
    }

print(f'\nData loaded for {len(results)} / {len(available)} systems.')

In [ ]:
"""
CELL 5 — FIGURE 1: Convergence quality

WHERE: First analysis cell; must pass before any other result is trusted.

WHAT each panel shows and WHY
-------------------------------
Panel A — Cumulative helix fraction f(t)
  WHAT: Running mean of the binary helix state over time.
  WHY: If the trajectory has not sampled enough transitions the running mean
       will still be drifting at t = 1 µs, and the reported f_helix will
       be meaningless.  We require the trace to plateau and for the 1st-half
       and 2nd-half means to agree within Δ < 0.02.

Panel B — Block Standard Error (BSE)
  WHAT: Statistical uncertainty of f_helix as a function of block size.
  WHY: The BSE rises with block size, reflecting the true correlation length
       of the trajectory.  It plateaus when blocks are longer than the
       autocorrelation time.  The plateau value is the true uncertainty;
       we require BSE_plateau < 0.01 for the result to be publishable.
  WHAT a still-rising BSE means: the trajectory is shorter than ~5× τ_corr
       and the sampling is insufficient.

Panel C — Split-trajectory 1D FES
  WHAT: FES computed from the first half (0–500 ns) and second half (500–1000 ns)
       of the trajectory.
  WHY: If both halves show the same basin locations and barrier height, the
       simulation has sampled both the helix and the coil basins reversibly.
       A systematic shift between the two halves indicates the trajectory
       has not yet equilibrated.

PASS criteria (Grossfield & Zuckerman 2009):
  |f_helix(1st half) − f_helix(2nd half)| < 0.02    [Panel A annotation]
  BSE plateau value < 0.01                            [Panel B annotation]
"""

for key, r in results.items():
    s    = r['system']
    n    = r['n_frames']
    mid  = n // 2
    t    = r['time_ns']
    rmsd = r['rmsd_nm']
    st   = r['state_primary']
    kin  = r['kinetics']

    fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))
    fig.suptitle(f"{s['label']} — Convergence", fontsize=12, fontweight='bold')

    # ── A: Cumulative helix fraction ──────────────────────────────────────────
    cum_f = np.cumsum(st.astype(float)) / (np.arange(n) + 1)
    fh1   = st[:mid].mean()
    fh2   = st[mid:].mean()
    delta = abs(fh1 - fh2)

    axes[0].plot(t, cum_f, color=s['color'], lw=1.2, label='Cumulative f_helix')
    axes[0].axhline(kin['f_helix'], color='k', ls='--', lw=1.0,
                    label=f'Overall = {kin["f_helix"]:.3f}')
    axes[0].axvline(t[mid], color='grey', ls=':', lw=0.8, label='Midpoint')

    # Annotations: clamp y-position to stay within axes
    y1 = min(fh1 + 0.05, 0.90)
    y2 = min(fh2 + 0.05, 0.90)
    axes[0].annotate(f'1st: {fh1:.3f}', xy=(t[mid // 4], y1), fontsize=9, color='navy')
    axes[0].annotate(f'2nd: {fh2:.3f}', xy=(t[mid + mid // 4], y2), fontsize=9, color='darkred')

    axes[0].set_xlabel('Time (ns)')
    axes[0].set_ylabel('Cumulative f_helix')
    axes[0].set_title('Helix fraction convergence')
    axes[0].set_ylim(0, 1)
    axes[0].legend(fontsize=9)
    conv_str = 'CONVERGED' if delta < 0.02 else 'NOT CONVERGED'
    axes[0].text(0.02, 0.05, f'Δ = {delta:.3f}  [{conv_str}]',
                 transform=axes[0].transAxes, fontsize=9,
                 color='green' if delta < 0.02 else 'red')

    # ── B: Block Standard Error ───────────────────────────────────────────────
    bse_bs   = r['bse_bs']
    bse_vals = r['bse_vals']
    if len(bse_vals) > 0:
        axes[1].semilogx(bse_bs, bse_vals, 'o-', color=s['color'], markersize=4, lw=1.5)
        axes[1].axhline(0.01, color='grey', ls='--', lw=1, label='Target BSE = 0.01')
        plateau = bse_vals[-1]
        axes[1].text(0.02, 0.95,
                     f'Plateau BSE ≈ {plateau:.4f}\n'
                     f'({"OK" if plateau < 0.01 else "INSUFFICIENT SAMPLING"})',
                     transform=axes[1].transAxes, fontsize=9, va='top',
                     color='green' if plateau < 0.01 else 'red')
    axes[1].set_xlabel('Block size (ns)')
    axes[1].set_ylabel('BSE (f_helix)')
    axes[1].set_title('Block Standard Error')
    axes[1].legend(fontsize=9)

    # ── C: Split-trajectory 1D FES ────────────────────────────────────────────
    for half_rmsd, lbl, ls in [
        (rmsd[:mid], '1st half (0 – 500 ns)',   '-'),
        (rmsd[mid:], '2nd half (500 – 1000 ns)', '--'),
    ]:
        xc, G1d = fes_1d(half_rmsd * 10, bins=FES_BINS_1D)
        axes[2].plot(xc, G1d, ls=ls, lw=1.8, label=lbl)
    axes[2].set_xlabel('RMSD to α-helix (Å)')
    axes[2].set_ylabel('ΔG (kcal/mol)')
    axes[2].set_title('FES: 1st vs 2nd half (overlap = convergence)')
    axes[2].set_ylim(-0.1, FES_CLIP)
    axes[2].legend(fontsize=9)

    plt.tight_layout()
    out = f"{s['prefix']}_{s['conf']}_convergence.png"
    plt.savefig(out, dpi=150, bbox_inches='tight')
    plt.show()
    print(f'  1st f_helix={fh1:.3f}  2nd f_helix={fh2:.3f}  Δ={delta:.3f}')
    print(f'  Saved: {out}\n')

In [ ]:
"""
CELL 6 — FIGURE 2: Order parameter time series with state overlay

WHERE: Second analysis cell; the visual inspection step.

WHAT each panel shows and WHY
-------------------------------
Panel A — Backbone RMSD to ideal helix (Å)
  WHY RMSD as the top panel: it is the primary reaction coordinate; the
  red/blue dashed lines show the hysteresis zone used for state classification.
  Visually confirms that the threshold choice is appropriate (transitions
  should cross both lines cleanly, not oscillate in the zone).

Panel B — Number of helical H-bonds (N_HB)
  WHY: Directly reflects helix content; discrete integer scale makes the
  two states visually obvious (full helix: 5–6 bonds; coil: 0–2 bonds).
  The dashed line shows the 50% threshold for the HB criterion.

Panel C — DSSP α-helix fraction
  WHY: Independent of the analyst's thresholds; shows whether the algorithm
  agrees with RMSD and N_HB.  Any systematic disagreement signals a structural
  anomaly (e.g., 3₁₀-helix, which DSSP would label 'G', not 'H').

State overlay and transition markers
  Red shading: frames classified as helix by the primary (RMSD) criterion.
  Green verticals: C→H transitions (fold events).
  Red verticals: H→C transitions (unfold events).
  These markers are the direct, irrefutable proof of the transition count.
"""

for key, r in results.items():
    s   = r['system']
    t   = r['time_ns']
    kin = r['kinetics']
    st  = r['state_primary']

    fig, axes = plt.subplots(3, 1, figsize=(14, 9), sharex=True)
    fig.suptitle(
        f"{s['label']} — Order parameter time series",
        fontsize=12, fontweight='bold'
    )

    for ax in axes:
        ax.fill_between(t, 0, 1, where=st,
                        alpha=0.12, color='#e63946',
                        transform=ax.get_xaxis_transform(),
                        label='Helix (RMSD criterion)')

    # Panel A: RMSD
    axes[0].plot(t, r['rmsd_nm'] * 10, color=s['color'], lw=0.5, alpha=0.85)
    axes[0].axhline(RMSD_HELIX_NM * 10, color='red',  ls='--', lw=1.2,
                    label=f'Helix bound ({RMSD_HELIX_NM*10:.0f} Å) — hysteresis lower')
    axes[0].axhline(RMSD_COIL_NM  * 10, color='blue', ls='--', lw=1.2,
                    label=f'Coil bound ({RMSD_COIL_NM*10:.0f} Å) — hysteresis upper')
    axes[0].set_ylabel('RMSD to α-helix (Å)')
    axes[0].legend(loc='upper right', fontsize=8)

    # Panel B: H-bond count
    thresh_hb = s['max_hb'] * HB_HELIX_FRAC
    axes[1].plot(t, r['hb_count'], color=s['color'], lw=0.5, alpha=0.85)
    axes[1].axhline(thresh_hb, color='green', ls='--', lw=1.2,
                    label=f'Helix threshold: {thresh_hb:.0f} H-bonds (50% of {s["max_hb"]})')
    axes[1].set_ylabel('Helical H-bonds (N_HB)')
    axes[1].set_ylim(-0.5, s['max_hb'] + 0.5)
    axes[1].set_yticks(range(s['max_hb'] + 1))
    axes[1].legend(loc='upper right', fontsize=8)

    # Panel C: DSSP fraction
    axes[2].plot(t, r['dssp_per_frame'], color=s['color'], lw=0.5, alpha=0.85)
    axes[2].axhline(DSSP_HELIX_FRAC, color='purple', ls='--', lw=1.2,
                    label=f'Helix threshold: {DSSP_HELIX_FRAC:.0%} residues = H')
    axes[2].set_ylabel('DSSP α-helix fraction')
    axes[2].set_ylim(-0.05, 1.05)
    axes[2].legend(loc='upper right', fontsize=8)
    axes[2].set_xlabel('Time (ns)')

    # Transition markers
    for idx in kin['fold_frames']:
        for ax in axes:
            ax.axvline(t[idx], color='#2d6a4f', lw=0.4, alpha=0.5)
    for idx in kin['unfold_frames']:
        for ax in axes:
            ax.axvline(t[idx], color='#d62828', lw=0.4, alpha=0.5)

    summary = (
        f"f_helix={kin['f_helix']:.3f}  "
        f"N_fold={kin['n_fold']}  N_unfold={kin['n_unfold']}  "
        f"τ_H={kin['tau_H_ns']:.1f} ns  τ_C={kin['tau_C_ns']:.1f} ns  "
        f"τ_eq={kin['tau_eq_ns']:.1f} ns"
    )
    fig.text(0.01, 0.01, summary, fontsize=9, color='#333333')

    plt.tight_layout(rect=[0, 0.03, 1, 1])
    out = f"{s['prefix']}_{s['conf']}_timeseries.png"
    plt.savefig(out, dpi=150, bbox_inches='tight')
    plt.show()
    print(f'  Saved: {out}\n')

In [ ]:
"""
CELL 7 — FIGURE 3: Triple-criterion agreement

WHERE: Validation of the state assignment before any kinetics are reported.

WHAT it shows and WHY
----------------------
Four panels stacked vertically for the same time axis:
  Panels 1–3: binary helix/coil from RMSD, H-bond, and DSSP criteria.
  Panel 4:    Agreement score = sum of the three binary signals (0–3).

WHY this plot is the 'irrefutable scientific validation':
  If the three criteria are merely noisy versions of the same underlying
  signal, they will agree > 90% of the time.  If they disagree frequently
  it indicates the peptide is spending significant time in intermediate
  states (e.g., partial helix, 3₁₀-helix) that are genuinely ambiguous —
  in which case a two-state model is oversimplified.

WHAT the 'contested' fraction means:
  Frames with score = 1 or 2 are contested: two criteria disagree.
  For a good two-state system this should be < 10% of frames.
  High contested fraction means the FES has a broad barrier or multiple
  intermediate states, and the kinetic analysis may be unreliable.
"""

for key, r in results.items():
    s     = r['system']
    t     = r['time_ns']
    score = r['agree_score']

    fig, axes = plt.subplots(4, 1, figsize=(14, 9), sharex=True)
    fig.suptitle(
        f"{s['label']} — Triple-criterion state assignment",
        fontsize=12, fontweight='bold'
    )

    data   = [r['state_rmsd'].astype(int),
              r['state_hb'].astype(int),
              r['state_dssp'].astype(int),
              score]
    labels = ['RMSD criterion (hysteresis)',
              'H-bond criterion (N_HB ≥ 50%)',
              'DSSP criterion (Kabsch-Sander ≥ 50%)',
              'Agreement score (0 = coil, 3 = helix)']
    colors = ['#e63946', '#457b9d', '#8338ec', '#06d6a0']

    for i, (ax, y, c, lbl) in enumerate(zip(axes, data, colors, labels)):
        ax.fill_between(t, y, step='mid', alpha=0.65, color=c)
        ax.set_ylabel(lbl, fontsize=9)
        if i < 3:
            ax.set_ylim(-0.1, 1.4)
            ax.set_yticks([0, 1])
            ax.set_yticklabels(['Coil', 'Helix'], fontsize=8)
        else:
            ax.set_ylim(-0.1, 3.5)
            ax.set_yticks([0, 1, 2, 3])
            ax.axhline(2.5, color='k', ls=':', lw=0.8, alpha=0.5,
                       label='Majority helix (score ≥ 3)')
            ax.legend(fontsize=8)

    axes[-1].set_xlabel('Time (ns)')

    unan_helix = (score == 3).mean()
    unan_coil  = (score == 0).mean()
    contested  = ((score > 0) & (score < 3)).mean()

    summary = (
        f"Unanimous helix: {unan_helix:.1%}  |  "
        f"Unanimous coil: {unan_coil:.1%}  |  "
        f"Contested: {contested:.1%}  "
        f"({'OK' if contested < 0.10 else 'CAUTION: intermediate states present'})"
    )
    fig.text(0.01, 0.01, summary, fontsize=9, color='#333333')

    plt.tight_layout(rect=[0, 0.03, 1, 1])
    out = f"{s['prefix']}_{s['conf']}_triple_criterion.png"
    plt.savefig(out, dpi=150, bbox_inches='tight')
    plt.show()
    print(summary)
    print(f'  Saved: {out}\n')

In [ ]:
"""
CELL 8 — FIGURE 4: Conformational populations

WHERE: After state validation; characterises where helix content comes from
       at a residue-by-residue level.

WHAT each panel shows and WHY
-------------------------------
Panel A — Per-residue helix fraction (torsion criterion, Best & Hummer 2009)
  WHAT: Bar chart; each bar = fraction of frames where that residue's (φ, ψ)
        falls in the α-helical basin (−100° < φ < −20°, −80° < ψ < 0°).
  WHY: Residue-level helix propensity is the key observable distinguishing
       A10 and A8Q2: the Gln residues at positions 3 and 8 of A8Q2 have lower
       helix propensity than Ala (Kuczera Crystals 2024), which should appear
       as a dip at those positions.  This comparison validates or refutes
       whether the CV-transferred model captures the mutation effect.
  COLOUR SCALE: RdYlGn (red = 0, green = 1) matches the FES colour convention
       where red = unfavourable (coil) and green = favourable (helix).

Panel B — Per-bond H-bond occupancy
  WHAT: Bar chart; each bar = fraction of frames where H-bond pair i→(i+4)
        is formed (H···O distance < 3.6 Å).  x-axis labels are 'residue_i →
        residue_{i+4}'.
  WHY: The occupancy pattern confirms the peptide is forming a true α-helix
       (all central bonds ~equal occupancy) rather than a 3₁₀-helix
       (i→i+3 pattern) or frayed structure (only 1–2 bonds near the centre).
       Terminal bonds have lower occupancy due to end-fraying — this is the
       expected physical picture for a capped 10-mer.

Panel C — Per-residue DSSP helix fraction
  WHAT: Same bar chart as Panel A but from DSSP algorithm.
  WHY: Cross-validates Panel A; differences between torsion-based and
       DSSP-based per-residue helicity reveal residues that have the right
       torsion angles but weak H-bonds (or vice versa), an indicator of
       structural strain at that position.
"""

for key, r in results.items():
    s         = r['system']
    phi       = r['phi']
    per_res   = r['per_res_frac']
    hb_occ    = r['hb_occupancy']    # (n_pairs,) float
    n_pairs   = r['n_hb_pairs']
    dssp_pr   = r['dssp_per_res']

    if phi is None or per_res is None:
        print(f"  Skipping {s['label']} — torsion data unavailable.")
        continue

    n_phi_res = phi.shape[1]
    ncols_fig = 3 if (dssp_pr is not None) else 2
    fig, axes = plt.subplots(1, ncols_fig, figsize=(5.5 * ncols_fig, 4.5))
    if ncols_fig == 2:
        axes = list(axes)
    else:
        axes = list(axes)
    fig.suptitle(f"{s['label']} — Conformational populations",
                 fontsize=12, fontweight='bold')

    # Normalise per_res to [0, 1] for the colourmap (values are already [0,1])
    norm = plt.Normalize(0, 1)
    cmap = plt.cm.RdYlGn

    # ── A: Per-residue helix fraction (torsion) ───────────────────────────────
    ax = axes[0]
    x_pos = np.arange(n_phi_res)
    ax.bar(x_pos, per_res, color=cmap(norm(per_res)), edgecolor='k', lw=0.5)
    ax.axhline(0.5, color='k', ls='--', lw=1, label='50% threshold')
    ax.set_xlabel('Residue (internal, excluding caps)')
    ax.set_ylabel('Helix fraction (φ/ψ criterion)')
    ax.set_title('Per-residue helix propensity\n(Best & Hummer 2009)')
    ax.set_ylim(0, 1.05)
    ax.set_xticks(x_pos)
    ax.set_xticklabels([f'R{i+1}' for i in range(n_phi_res)], fontsize=9)
    ax.legend(fontsize=9)
    sm = plt.cm.ScalarMappable(cmap=cmap, norm=norm)
    sm.set_array([])
    plt.colorbar(sm, ax=ax, shrink=0.8, label='Helix fraction')

    # ── B: Per-bond H-bond occupancy ──────────────────────────────────────────
    ax2 = axes[1]
    if len(hb_occ) > 0:
        x_hb = np.arange(len(hb_occ))
        ax2.bar(x_hb, hb_occ, color=cmap(norm(hb_occ)), edgecolor='k', lw=0.5)
        ax2.axhline(0.5, color='k', ls='--', lw=1, label='50% occupancy')
        ax2.set_xlabel('H-bond pair (i → i+4)')
        ax2.set_ylabel('Occupancy (fraction of frames)')
        ax2.set_title(f'Per-bond H-bond occupancy\n(distance < 3.6 Å; Kuczera 2021)')
        ax2.set_ylim(0, 1.05)
        ax2.set_xticks(x_hb)
        ax2.set_xticklabels([f'HB{i+1}' for i in range(len(hb_occ))],
                             fontsize=8, rotation=30)
        ax2.legend(fontsize=9)
        sm2 = plt.cm.ScalarMappable(cmap=cmap, norm=norm)
        sm2.set_array([])
        plt.colorbar(sm2, ax=ax2, shrink=0.8, label='Occupancy')
    else:
        ax2.text(0.5, 0.5, 'No H-bond pairs\nidentified',
                 ha='center', va='center', transform=ax2.transAxes, fontsize=11)
        ax2.axis('off')

    # ── C: DSSP per-residue ───────────────────────────────────────────────────
    if ncols_fig == 3:
        ax3 = axes[2]
        n_dssp = len(dssp_pr)
        x_d    = np.arange(n_dssp)
        ax3.bar(x_d, dssp_pr, color=cmap(norm(dssp_pr)), edgecolor='k', lw=0.5)
        ax3.axhline(0.5, color='k', ls='--', lw=1)
        ax3.set_xlabel('Residue')
        ax3.set_ylabel('DSSP helix fraction')
        ax3.set_title('Per-residue DSSP helix\n(Kabsch-Sander 1983)')
        ax3.set_ylim(0, 1.05)
        ax3.set_xticks(x_d)
        ax3.set_xticklabels([f'R{i+1}' for i in range(n_dssp)], fontsize=9)
        sm3 = plt.cm.ScalarMappable(cmap=cmap, norm=norm)
        sm3.set_array([])
        plt.colorbar(sm3, ax=ax3, shrink=0.8, label='DSSP helix fraction')

    plt.tight_layout()
    out = f"{s['prefix']}_{s['conf']}_populations.png"
    plt.savefig(out, dpi=150, bbox_inches='tight')
    plt.show()
    print(f'  Per-residue (torsion): {np.array2string(per_res, precision=3)}')
    if len(hb_occ) > 0:
        print(f'  Per-bond occupancy: {np.array2string(hb_occ, precision=3)}')
    print(f'  Saved: {out}\n')

In [ ]:
"""
CELL 9 — FIGURE 5: 2D and 1D Free Energy Surfaces

WHERE: Core thermodynamic characterisation; ground truth for CV-transfer validation.

WHAT each panel shows and WHY
-------------------------------
Left — 2D FES: RMSD (Å) × N_HB
  WHAT: Joint probability density mapped to free energy.
  WHY TWO axes: RMSD captures global shape; N_HB captures local H-bond
       pattern.  In a perfect two-state system the two minima lie in the
       lower-right (low RMSD, high N_HB = helix) and upper-left (high RMSD,
       low N_HB = coil) corners.  Any diagonal, off-axis density indicates
       intermediate states.
  WHY contour lines at 1, 2, 3 kcal/mol: at 320 K, kBT ≈ 0.64 kcal/mol,
       so these lines correspond to exp(−1/0.64) ≈ 21%, exp(−2/0.64) ≈ 4%,
       exp(−3/0.64) ≈ 0.8% of maximum population — showing the core basin,
       the shoulders, and the accessible tail.

Right — 1D FES along N_HB
  WHAT: Marginal free energy as a function of H-bond count only.
  WHY: This is the most direct comparison with experimental helix content
       from CD spectroscopy (which measures total helix fraction, not shape).
       The two-state character — two minima separated by a maximum — justifies
       the two-state kinetic model.  A single minimum indicates folded-only
       or unfolded-only within the simulation window.
  WHY integer bins: N_HB is a discrete count (0–6); binning by N_HB + 1 gives
       one bin per integer value, preserving the physical discreteness.
"""

for key, r in results.items():
    s        = r['system']
    rmsd_A   = r['rmsd_nm'] * 10        # nm → Å for display
    hb_float = r['hb_count'].astype(float)

    fig, axes = plt.subplots(1, 2, figsize=(13, 5))
    fig.suptitle(f"{s['label']} — Free Energy Surface", fontsize=12, fontweight='bold')

    # ── Left: 2D FES ──────────────────────────────────────────────────────────
    xi, yi, G2d = fes_2d(rmsd_A, hb_float, bins=FES_BINS_2D)
    levels = np.linspace(0, FES_CLIP, 25)
    cf = axes[0].contourf(xi, yi, G2d, levels=levels, cmap='RdYlGn_r', extend='max')
    cs = axes[0].contour(xi, yi, G2d, levels=[1.0, 2.0, 3.0],
                         colors='k', linewidths=0.6, alpha=0.5)
    axes[0].clabel(cs, fmt='%1.0f kcal/mol', fontsize=7)
    plt.colorbar(cf, ax=axes[0], label='ΔG (kcal/mol)')
    axes[0].set_xlabel('RMSD to α-helix (Å)')
    axes[0].set_ylabel('# Helical H-bonds (N_HB)')
    axes[0].set_title('2D FES: RMSD × N_HB\n(Boltzmann inversion, Tavernelli 2003)')
    axes[0].axvline(RMSD_HELIX_NM * 10, color='red',  ls='--', lw=1.0,
                    alpha=0.8, label='Helix RMSD bound')
    axes[0].axvline(RMSD_COIL_NM  * 10, color='blue', ls='--', lw=1.0,
                    alpha=0.8, label='Coil RMSD bound')
    axes[0].axhline(s['max_hb'] * HB_HELIX_FRAC, color='green', ls=':', lw=1.0,
                    alpha=0.8, label='HB threshold')
    axes[0].legend(fontsize=8)

    # ── Right: 1D FES along N_HB ──────────────────────────────────────────────
    xc_hb, G_hb = fes_1d(hb_float, bins=s['max_hb'] + 1)
    axes[1].plot(xc_hb, G_hb, 'o-', color=s['color'], lw=2, markersize=7,
                 label='FES(N_HB)')
    axes[1].axvline(s['max_hb'] * HB_HELIX_FRAC, color='green', ls='--', lw=1.2,
                    label=f'Helix threshold ({s["max_hb"]*HB_HELIX_FRAC:.0f} bonds)')
    axes[1].set_xlabel('Number of helical H-bonds (N_HB)')
    axes[1].set_ylabel('ΔG (kcal/mol)')
    axes[1].set_title('1D FES along N_HB\n(marginal of 2D; direct comparison with CD)')
    axes[1].set_ylim(-0.1, FES_CLIP)
    axes[1].set_xticks(range(s['max_hb'] + 1))
    axes[1].legend(fontsize=9)

    valid = ~np.isnan(G_hb)
    if valid.sum() > 0:
        imin = np.nanargmin(G_hb)
        axes[1].annotate(
            f'Global min\nN_HB={xc_hb[imin]:.1f}\nΔG={G_hb[imin]:.2f}',
            xy=(xc_hb[imin], G_hb[imin]),
            xytext=(xc_hb[imin] + 0.8, G_hb[imin] + 0.6),
            arrowprops=dict(arrowstyle='->', color='k'), fontsize=8
        )

    plt.tight_layout()
    out = f"{s['prefix']}_{s['conf']}_FES.png"
    plt.savefig(out, dpi=150, bbox_inches='tight')
    plt.show()
    print(f'  Saved: {out}\n')

In [ ]:
"""
CELL 10 — FIGURE 6: Per-residue Ramachandran PMF

WHERE: Deepest structural characterisation; links simulation to structural
       biology and provides input for TICA feature selection.

WHAT it shows and WHY
----------------------
Each sub-panel: 2D histogram of (φ, ψ) for one residue, converted to
  ΔG(φ, ψ) = −kBT ln[P(φ, ψ) / P_max]

WHY per-residue instead of aggregated?
  An aggregated Ramachandran plot averages over all residues; it looks
  like a generic polypeptide plot and cannot show residue-specific effects.
  For A8Q2, Gln residues (positions 3 and 8 in the sequence) should have
  a broader, shallower α-helical basin than Ala due to Gln's side-chain
  entropy cost for helix formation.  The per-residue PMF directly reveals
  this positional effect.

WHY 8° bins (45 bins across ±180°)?
  Each residue contributes n_frames points to the 2D histogram.  For
  10 000 frames and 44×44 = 1936 bins, the expected count per bin in a
  populated region is ~5 — sufficient for stable Boltzmann inversion.
  Finer bins would give noisier FES; coarser bins would smear out real features.

WHY clip at 4.0 kcal/mol?
  At 4 kcal/mol the population is exp(−4/0.64) ≈ 0.001 × P_max — essentially
  zero.  Regions above this are either sampling artefacts or genuinely
  inaccessible and would dominate the colour scale if not clipped.

White star: ideal α-helix centre (φ = −62°, ψ = −41°, Kuczera 2021).
White dashed lines: helix-basin boundaries.
A residue with a deep basin centred near the star is strongly helical.
A residue with a broad or shifted basin has reduced helix propensity.
"""

for key, r in results.items():
    s   = r['system']
    phi = r['phi']
    psi = r['psi']

    if phi is None:
        print(f"  Skipping {s['label']} — torsion data unavailable.")
        continue

    n_res = phi.shape[1]
    ncols = 4
    nrows = int(np.ceil(n_res / ncols))

    # If nrows == 1, plt.subplots returns a 1D array; if nrows > 1 it returns 2D.
    # Calling squeeze=False ensures we always get a 2D array that can be flattened.
    fig, axes = plt.subplots(nrows, ncols, figsize=(4.0 * ncols, 3.5 * nrows),
                             squeeze=False)
    axes_flat = axes.flatten()
    fig.suptitle(f"{s['label']} — Per-residue Ramachandran PMF",
                 fontsize=12, fontweight='bold')

    edges   = np.linspace(-180, 180, 45)   # 8° bins
    centres = 0.5 * (edges[:-1] + edges[1:])

    for i in range(n_res):
        ax    = axes_flat[i]
        phi_d = np.degrees(phi[:, i])
        psi_d = np.degrees(psi[:, i])

        H, _, _ = np.histogram2d(phi_d, psi_d, bins=[edges, edges], density=False)
        H       = H.astype(float)
        with np.errstate(divide='ignore', invalid='ignore'):
            G = np.where(H > 0, -kBT_kcal * np.log(H / H.max()), np.nan)
        G = np.clip(G, 0.0, 4.0)

        xi_r, yi_r = np.meshgrid(centres, centres, indexing='ij')
        ax.contourf(xi_r, yi_r, G,
                    levels=np.linspace(0, 4.0, 17),
                    cmap='RdYlGn_r', extend='max')

        ax.axvline(PHI_HELIX_DEG, color='white', ls='--', lw=0.8, alpha=0.7)
        ax.axhline(PSI_HELIX_DEG, color='white', ls='--', lw=0.8, alpha=0.7)
        ax.scatter([PHI_HELIX_DEG], [PSI_HELIX_DEG],
                   color='white', s=40, zorder=5, marker='*')
        ax.set_xlim(-180, 180)
        ax.set_ylim(-180, 180)
        ax.set_title(f'Res {i+1}', fontsize=9)
        ax.set_xticks([-120, 0, 120])
        ax.set_yticks([-120, 0, 120])
        ax.tick_params(labelsize=7)
        if i % ncols == 0:
            ax.set_ylabel('ψ (°)', fontsize=8)
        if i >= (nrows - 1) * ncols:
            ax.set_xlabel('φ (°)', fontsize=8)

    for j in range(n_res, len(axes_flat)):
        axes_flat[j].set_visible(False)

    # Shared colourbar — pass a list of all used axes
    sm = plt.cm.ScalarMappable(cmap='RdYlGn_r', norm=plt.Normalize(0, 4.0))
    sm.set_array([])
    fig.colorbar(sm, ax=axes_flat[:n_res].tolist(), shrink=0.6,
                 label='ΔG (kcal/mol)')

    plt.tight_layout()
    out = f"{s['prefix']}_{s['conf']}_Ramachandran.png"
    plt.savefig(out, dpi=150, bbox_inches='tight')
    plt.show()
    print(f'  Saved: {out}\n')

In [ ]:
"""
CELL 11 — FIGURE 7: Transition kinetics

WHERE: Quantitative kinetics; the primary data for literature comparison
       and enhanced sampling validation.

WHAT each panel shows and WHY
-------------------------------
Panel A — Helix sojourn time distribution
  WHAT: Histogram of how long each continuous helix-state period lasts.
  WHY: For a Markovian two-state system the sojourn time distribution is
       exponential: P(τ) = k_u · exp(−k_u · τ).  A non-exponential
       distribution (e.g., bimodal, heavy-tailed) signals memory effects,
       i.e. the system is not strictly two-state.  The mean <τ_H> = 1/k_u
       is marked with a dashed line.

Panel B — Coil sojourn time distribution
  WHAT: Same for coil-state periods; mean <τ_C> = 1/k_f marked.
  WHY: Together with Panel A this directly shows which state is kinetically
       'sticky' (long sojourns).  For polyalanine the coil state has shorter
       sojourns (faster folding) than helix (slow unfolding), consistent with
       coil being the thermodynamically preferred state for ALA8.

Panel C — Autocorrelation function (ACF) + biexponential fit
  WHAT: ACF of N_HB(t) and the fitted biexponential decay.
  WHY: Provides an independent, model-free estimate of τ_relax that does not
       require defining discrete states or transition events.  The τ_slow from
       the biexponential fit should agree with τ_eq = 1/(k_f + k_u) from
       transition counting within a factor of ~2.  Disagreement indicates
       that either the state classification has artefactual transitions or
       the trajectory is too short to reliably fit the ACF.
  The red vertical line on the ACF plot shows τ_eq from counting — the
  cross-validation visualised directly.

Panel D — Summary statistics table
  Includes the Arrhenius-corrected literature comparison (Kuczera 2021, 300 K)
  scaled to 320 K using Eₐ ≈ 10 kcal/mol so the reader can immediately assess
  whether the simulations are producing physically reasonable kinetics.
"""

for key, r in results.items():
    s   = r['system']
    kin = r['kinetics']

    fig = plt.figure(figsize=(15, 8))
    gs  = GridSpec(2, 3, figure=fig, hspace=0.45, wspace=0.35)
    fig.suptitle(f"{s['label']} — Transition kinetics", fontsize=12, fontweight='bold')

    # ── A: Helix sojourn distribution ─────────────────────────────────────────
    ax_A = fig.add_subplot(gs[0, 0])
    h_soj = kin['h_sojourns']
    if len(h_soj) > 2:
        ax_A.hist(h_soj, bins='auto', color='#e63946', alpha=0.75,
                  edgecolor='white', lw=0.5)
        ax_A.axvline(kin['tau_H_ns'], color='k', ls='--', lw=1.5,
                     label=f'⟨τ_H⟩ = {kin["tau_H_ns"]:.1f} ns')
        ax_A.set_xlabel('Helix sojourn duration (ns)')
        ax_A.set_ylabel('Count')
        ax_A.set_title('Helix residence times\n(exponential ≈ Markovian)')
        ax_A.legend(fontsize=9)
    else:
        ax_A.text(0.5, 0.5, f'Too few\ntransitions\n(N={len(h_soj)})',
                  ha='center', va='center', transform=ax_A.transAxes, fontsize=12)
        ax_A.axis('off')

    # ── B: Coil sojourn distribution ──────────────────────────────────────────
    ax_B = fig.add_subplot(gs[0, 1])
    c_soj = kin['c_sojourns']
    if len(c_soj) > 2:
        ax_B.hist(c_soj, bins='auto', color='#457b9d', alpha=0.75,
                  edgecolor='white', lw=0.5)
        ax_B.axvline(kin['tau_C_ns'], color='k', ls='--', lw=1.5,
                     label=f'⟨τ_C⟩ = {kin["tau_C_ns"]:.1f} ns')
        ax_B.set_xlabel('Coil sojourn duration (ns)')
        ax_B.set_ylabel('Count')
        ax_B.set_title('Coil residence times\n(exponential ≈ Markovian)')
        ax_B.legend(fontsize=9)
    else:
        ax_B.text(0.5, 0.5, f'Too few\ntransitions\n(N={len(c_soj)})',
                  ha='center', va='center', transform=ax_B.transAxes, fontsize=12)
        ax_B.axis('off')

    # ── C: ACF + biexponential fit ─────────────────────────────────────────────
    ax_C = fig.add_subplot(gs[0, 2])
    lag  = r['acf_lag']
    acf  = r['acf_vals']
    ax_C.plot(lag, acf, color=s['color'], lw=0.8, alpha=0.8,
              label='ACF of N_HB order parameter')
    ax_C.axhline(0, color='k', lw=0.7, ls=':')
    if r['acf_fit'] is not None:
        popt, tau_slow, tau_fast, A_amp = r['acf_fit']
        t_fit = np.linspace(0, lag[-1], 500)
        y_fit = A_amp * np.exp(-t_fit / tau_slow) + (1 - A_amp) * np.exp(-t_fit / tau_fast)
        ax_C.plot(t_fit, y_fit, 'k--', lw=1.5,
                  label=f'Biexp fit\nτ_slow = {tau_slow:.1f} ns\nτ_fast = {tau_fast:.2f} ns')
        if not np.isnan(kin['tau_eq_ns']):
            ax_C.axvline(kin['tau_eq_ns'], color='red', ls=':', lw=1.2,
                         label=f'τ_eq (counting) = {kin["tau_eq_ns"]:.1f} ns')
    ax_C.set_xlabel('Lag time (ns)')
    ax_C.set_ylabel('ACF')
    ax_C.set_title('Autocorrelation of N_HB\n(τ_slow ≈ τ_eq = cross-validation)')
    ax_C.set_ylim(-0.2, 1.05)
    ax_C.legend(fontsize=8)

    # ── D: Summary statistics ──────────────────────────────────────────────────
    ax_D = fig.add_subplot(gs[1, :])
    ax_D.axis('off')

    def _fmt(v):
        return f'{float(v):.4g}' if (v is not None and not np.isnan(float(v))) else 'N/A'

    stats = (
        f"THERMODYNAMICS\n"
        f"  f_helix = {kin['f_helix']:.3f}   ΔG_fold = {_fmt(kin['dG_kcal'])} kcal/mol\n"
        f"  (ΔG_fold < 0 → helix more stable;  ΔG_fold > 0 → coil more stable)\n"
        f"\nTRANSITION COUNTING (MFPT)"
        f"  [N_fold = {kin['n_fold']}  N_unfold = {kin['n_unfold']}]\n"
        f"  τ_H = {_fmt(kin['tau_H_ns'])} ns    k_u = {_fmt(kin['k_u_ns'])} ns⁻¹\n"
        f"  τ_C = {_fmt(kin['tau_C_ns'])} ns    k_f = {_fmt(kin['k_f_ns'])} ns⁻¹\n"
        f"  τ_eq (counting) = {_fmt(kin['tau_eq_ns'])} ns\n"
        f"  τ_slow (ACF fit) = {_fmt(r['tau_slow_ns'])} ns\n"
        f"\nLITERATURE — Kuczera & Jas 2021 (ALA8, CHARMM36m, 300 K):\n"
        f"  k_u ≈ (7.0–7.9)×10⁻² ns⁻¹    k_f ≈ (3.6–7.8)×10⁻³ ns⁻¹\n"
        f"  ΔG_fold ≈ 1.3–1.8 kcal/mol (coil stable)    τ_eq ≈ 12–13 ns at 300 K\n"
        f"\nARRHENIUS CORRECTION to 320 K (Eₐ ≈ 10 kcal/mol):\n"
        f"  speedup = exp(Eₐ/R × [1/300 − 1/320]) ≈ 5–8×  →  τ_eq(320 K) ≈ 1.5–3 ns"
    )
    ax_D.text(0.01, 0.95, stats, transform=ax_D.transAxes, fontsize=9.5,
              verticalalignment='top', fontfamily='monospace',
              bbox=dict(boxstyle='round', facecolor='#f8f9fa', alpha=0.9))

    out = f"{s['prefix']}_{s['conf']}_kinetics.png"
    plt.savefig(out, dpi=150, bbox_inches='tight')
    plt.show()
    print(f'  Saved: {out}\n')

In [ ]:
"""
CELL 12 — FIGURE 8: A10 vs A8Q2 comparison

WHERE: The direct comparison that defines the target for CV-transfer validation.

WHAT each panel shows and WHY
-------------------------------
Panels A/B — Side-by-side 2D FES (pooled trajectories)
  WHAT: FES of A10 and A8Q2 computed from pooled helix-start + extended-start
        trajectories (2 µs combined per peptide).
  WHY pool: Averaging the two independent trajectories reduces statistical
       noise by √2 and further validates convergence (if both starting
       conformations give the same FES, the pooled FES is reliable).
  WHAT to look for: If the two FES look nearly identical → CV transfer from
       A10 to A8Q2 should be trivially valid.  If the coil basin has shifted
       or the barrier height has changed → the CV transfer may fail or
       require fine-tuning.

Panel C — Per-residue helix fraction overlay
  WHAT: Mean per-residue helix fraction (φ/ψ criterion) from both starting
        conformations, plotted as a line for each peptide.
  WHY: This is the clearest observable effect of the Ala→Gln substitutions.
       Expected: A8Q2 has lower helix fraction at positions 3 and 8 (Gln
       sites) relative to A10, reflecting Gln's lower intrinsic helix
       propensity (Chakrabartty & Baldwin 1995 scale: Ala ΔΔG ≈ 0 kcal/mol,
       Gln ΔΔG ≈ +0.5 kcal/mol destabilising).

Panel D — 1D FES overlay along N_HB
  WHAT: Both peptides' 1D FES on the same axes.
  WHY: Directly shows whether the Gln substitutions shift the equilibrium
       towards coil (A8Q2 minimum at lower N_HB) or simply change the
       barrier height.  This is the ground truth the transferred CV model
       must reproduce.
"""

a10_res  = {k: v for k, v in results.items() if k[0] == 'A10'}
a8q2_res = {k: v for k, v in results.items() if k[0] == 'A8Q2'}

if not a10_res:
    print('No A10 results — skipping comparison.')
elif not a8q2_res:
    print('No A8Q2 results — skipping comparison.')
else:
    def pool(rd, field):
        return np.concatenate([v[field] for v in rd.values()])

    fig = plt.figure(figsize=(16, 11))
    gs  = GridSpec(2, 2, figure=fig, hspace=0.4, wspace=0.35)
    fig.suptitle('A10 vs A8Q2 — Ground-truth comparison (pooled trajectories)',
                 fontsize=13, fontweight='bold')

    # ── A/B: Side-by-side 2D FES ──────────────────────────────────────────────
    for col, (rd, title) in enumerate([
        (a10_res,  'Ac-Ala₁₀-NHMe (A10)'),
        (a8q2_res, 'Ac-Ala₈Gln₂-NHMe (A8Q2)'),
    ]):
        ax = fig.add_subplot(gs[0, col])
        rmsd_pool = pool(rd, 'rmsd_nm') * 10
        hb_pool   = pool(rd, 'hb_count').astype(float)

        xi, yi, G2d = fes_2d(rmsd_pool, hb_pool, bins=FES_BINS_2D)
        levels = np.linspace(0, FES_CLIP, 25)
        cf = ax.contourf(xi, yi, G2d, levels=levels, cmap='RdYlGn_r', extend='max')
        cs = ax.contour(xi, yi, G2d, levels=[1.0, 2.0, 3.0],
                        colors='k', linewidths=0.6, alpha=0.5)
        ax.clabel(cs, fmt='%1.0f', fontsize=7)
        plt.colorbar(cf, ax=ax, label='ΔG (kcal/mol)')
        ax.set_xlabel('RMSD to α-helix (Å)')
        ax.set_ylabel('# Helical H-bonds (N_HB)')
        ax.set_title(title)
        ax.axvline(RMSD_HELIX_NM * 10, color='red',  ls='--', lw=1, alpha=0.7)
        ax.axvline(RMSD_COIL_NM  * 10, color='blue', ls='--', lw=1, alpha=0.7)

    # ── C: Per-residue helix fraction overlay ─────────────────────────────────
    ax_C = fig.add_subplot(gs[1, 0])
    plotted = False
    for rd, pep_lbl, col in [
        (a10_res,  'A10',  '#e63946'),
        (a8q2_res, 'A8Q2', '#f4a261'),
    ]:
        prf_list = [v['per_res_frac'] for v in rd.values() if v['per_res_frac'] is not None]
        if prf_list:
            mean_prf = np.mean(prf_list, axis=0)
            x_pos    = np.arange(1, len(mean_prf) + 1)
            ax_C.plot(x_pos, mean_prf, 'o-', color=col, lw=2, markersize=7,
                      label=pep_lbl)
            plotted = True
    if plotted:
        ax_C.axhline(0.5, color='k', ls='--', lw=1, alpha=0.6, label='50% threshold')
        ax_C.set_xlabel('Residue index (internal)')
        ax_C.set_ylabel('Mean helix fraction (φ/ψ criterion)')
        ax_C.set_title('Per-residue helix propensity\n(dip at Gln positions expected for A8Q2)')
        ax_C.set_ylim(0, 1)
        ax_C.legend()
    else:
        ax_C.text(0.5, 0.5, 'No torsion data', ha='center', va='center',
                  transform=ax_C.transAxes)

    # ── D: 1D FES overlay along N_HB ──────────────────────────────────────────
    ax_D = fig.add_subplot(gs[1, 1])
    for rd, pep_lbl, col in [
        (a10_res,  'A10',  '#e63946'),
        (a8q2_res, 'A8Q2', '#f4a261'),
    ]:
        hb_pool = pool(rd, 'hb_count').astype(float)
        max_hb  = list(rd.values())[0]['system']['max_hb']
        xc, G1d = fes_1d(hb_pool, bins=max_hb + 1)
        ax_D.plot(xc, G1d, 'o-', color=col, lw=2, markersize=7, label=pep_lbl)
    ax_D.set_xlabel('Number of helical H-bonds (N_HB)')
    ax_D.set_ylabel('ΔG (kcal/mol)')
    ax_D.set_title('1D FES comparison\n(basin shift = effect of Gln substitutions)')
    ax_D.set_ylim(-0.1, FES_CLIP)
    ax_D.legend()

    plt.savefig('A10_vs_A8Q2_comparison.png', dpi=150, bbox_inches='tight')
    plt.show()
    print('Saved: A10_vs_A8Q2_comparison.png')

In [ ]:
"""
CELL 13 — FIGURE 9: Backbone torsion covariance and eigenvector overlap

WHERE: Final analysis cell; directly predicts whether the TICA/SPIB transfer
       will succeed before the transfer is even attempted.

WHAT it shows and WHY
----------------------
Sin/cos encoding of backbone torsions
  φ and ψ are angles; their mean and variance are undefined in the usual sense
  because of the ±180° discontinuity.  Encoding as (sin φ, cos φ, sin ψ, cos ψ)
  maps each angle to a point on the unit circle where the covariance matrix
  is well-defined and linear algebra applies.  This is the standard TICA feature
  vector (Pérez-Hernández 2013; Schwantes & Pande 2013).

Covariance matrix C and its eigenvectors
  C = X^T X / (N−1) where X is the mean-centred feature matrix.
  Eigenvectors of C are the Principal Components (PCs) of backbone motion.
  The PC with the largest eigenvalue captures the most variance — for a
  helix-coil system this should correspond to the helix-coil reaction
  coordinate (i.e., the projection of PC-1 should correlate strongly with N_HB).

WHY NOT use TICA eigenvectors directly?
  TICA requires defining a lag time and involves time-lagged covariance.
  For a pre-assessment of transferability, the zero-lag (PCA) covariance is
  sufficient: if the ZERO-LAG slow motions are different, the time-lagged
  ones certainly will be too.

Eigenvector overlap matrix O[i,j] = |v_i(A10) · v_j(A8Q2)|
  If A10 and A8Q2 have the same slow collective motions, their corresponding
  eigenvectors are parallel: the diagonal of O is close to 1.0 and
  off-diagonal elements are close to 0.

  O[0,0] close to 1 → PC-1 (helix-coil mode) is nearly identical in A10 and A8Q2
                    → TICA/SPIB CV trained on A10 will project A8Q2 trajectories
                       onto the same latent space; transfer is expected to work.

  O[0,0] << 1 → the dominant slow motion of A8Q2 is a different linear combination
                  of torsion angles than for A10 → the transferred CV will project
                  A8Q2 onto a physically incorrect space; the transferred FES and
                  kinetics will be wrong.

Pass criterion: O[0,0] > 0.70 and O[1,1] > 0.70 for the top-2 PCs.
  0.70 corresponds to a 45° angle between eigenvectors — a rough but widely
  used threshold for 'similar enough' in TICA transfer literature.
"""

a10_res  = {k: v for k, v in results.items() if k[0] == 'A10'}
a8q2_res = {k: v for k, v in results.items() if k[0] == 'A8Q2'}


def torsion_features(phi, psi):
    """
    Sin/cos encoding: (n_frames, 4 × n_res).
    Order: [sin φ₁..sin φₙ | cos φ₁..cos φₙ | sin ψ₁..sin ψₙ | cos ψ₁..cos ψₙ]
    """
    return np.concatenate([np.sin(phi), np.cos(phi), np.sin(psi), np.cos(psi)], axis=1)


def pca_eigenvecs(phi, psi, K=10):
    """
    Top-K eigenvectors of the torsion feature covariance matrix,
    sorted by descending eigenvalue.

    Returns (eigvals[:K], eigvecs[:, :K]) where eigvecs columns are eigenvectors.
    """
    X = torsion_features(phi, psi)
    X -= X.mean(axis=0)
    C = (X.T @ X) / (len(X) - 1)
    eigvals, eigvecs = np.linalg.eigh(C)   # eigh for symmetric matrix; more stable
    idx = np.argsort(eigvals)[::-1]
    return eigvals[idx][:K], eigvecs[:, idx][:, :K]


def pool_torsions(rd):
    phi_list = [v['phi'] for v in rd.values() if v['phi'] is not None]
    psi_list = [v['psi'] for v in rd.values() if v['psi'] is not None]
    if not phi_list:
        return None, None
    return np.vstack(phi_list), np.vstack(psi_list)


if a10_res and a8q2_res:
    phi_a10,  psi_a10  = pool_torsions(a10_res)
    phi_a8q2, psi_a8q2 = pool_torsions(a8q2_res)

    if phi_a10 is None or phi_a8q2 is None:
        print('Torsion data not available — skipping covariance analysis.')
    else:
        n_common = min(phi_a10.shape[1], phi_a8q2.shape[1])
        K = 6

        evals_a10,  evecs_a10  = pca_eigenvecs(phi_a10[:, :n_common],
                                                psi_a10[:, :n_common], K)
        evals_a8q2, evecs_a8q2 = pca_eigenvecs(phi_a8q2[:, :n_common],
                                                psi_a8q2[:, :n_common], K)

        # O[i,j] = |cosine(v_i(A10), v_j(A8Q2))|
        # evecs shape: (n_features, K); evecs.T is (K, n_features)
        # evecs_a10.T @ evecs_a8q2 gives (K, K) matrix of dot products
        overlap = np.abs(evecs_a10.T @ evecs_a8q2)

        fig, axes = plt.subplots(1, 3, figsize=(16, 5))
        fig.suptitle('Backbone torsion PCA — CV transfer readiness',
                     fontsize=12, fontweight='bold')

        # ── A: Eigenvalue spectra ──────────────────────────────────────────────
        x = np.arange(1, K + 1)
        axes[0].plot(x, evals_a10,  'o-', color='#e63946', lw=2, label='A10')
        axes[0].plot(x, evals_a8q2, 's-', color='#f4a261', lw=2, label='A8Q2')
        axes[0].set_xlabel('PC index')
        axes[0].set_ylabel('Eigenvalue (torsion variance)')
        axes[0].set_title('PC spectrum\n(similar shape → similar dynamics)')
        axes[0].legend()
        axes[0].set_xticks(x)

        # ── B: Overlap matrix heatmap ──────────────────────────────────────────
        im = axes[1].imshow(overlap, vmin=0, vmax=1, cmap='Blues', aspect='auto')
        plt.colorbar(im, ax=axes[1], label='|cos similarity|')
        axes[1].set_xticks(range(K))
        axes[1].set_yticks(range(K))
        axes[1].set_xticklabels([f'A8Q2-PC{i+1}' for i in range(K)],
                                  rotation=45, ha='right', fontsize=8)
        axes[1].set_yticklabels([f'A10-PC{i+1}' for i in range(K)], fontsize=8)
        axes[1].set_title(f'Eigenvector overlap O[i,j]\n(diagonal ≈ 1.0 = perfect transfer)')
        for i in range(K):
            for j in range(K):
                axes[1].text(j, i, f'{overlap[i,j]:.2f}',
                             ha='center', va='center', fontsize=7,
                             color='white' if overlap[i, j] > 0.6 else 'black')

        # ── C: Diagonal similarity bar chart ───────────────────────────────────
        diag = np.diag(overlap)
        colors_bar = ['#2d6a4f' if d > 0.7 else '#e63946' for d in diag]
        axes[2].bar(x, diag, color=colors_bar, edgecolor='k', lw=0.5)
        axes[2].axhline(0.7, color='k', ls='--', lw=1,
                        label='Threshold = 0.70')
        axes[2].set_xlabel('PC index')
        axes[2].set_ylabel('|cos(A10-PCᵢ, A8Q2-PCᵢ)|')
        axes[2].set_title('PC-matched similarity\n(green > 0.70 = transfer viable)')
        axes[2].set_ylim(0, 1.05)
        axes[2].set_xticks(x)
        axes[2].legend(fontsize=9)

        good = (diag[:2] > 0.7).all()
        verdict = (
            'GOOD: top-2 PCs similar (>0.70) → TICA/SPIB transfer expected valid'
            if good else
            'CAUTION: top-2 PCs dissimilar (<0.70) → CV transfer may require retraining'
        )
        fig.text(0.01, 0.01, verdict, fontsize=10, fontweight='bold',
                 color='#2d6a4f' if good else '#e63946')

        plt.tight_layout(rect=[0, 0.06, 1, 1])
        plt.savefig('torsion_covariance_overlap.png', dpi=150, bbox_inches='tight')
        plt.show()
        print(f'PC-1 overlap: {diag[0]:.3f}   PC-2 overlap: {diag[1]:.3f}')
        print(verdict)
        print('Saved: torsion_covariance_overlap.png')
else:
    print('Need both A10 and A8Q2 results for covariance comparison — skipping.')

In [ ]:
"""
CELL 14 — Summary table and CSV export

All kinetic and thermodynamic observables for all systems in one table.
Exported to CSV for inclusion in reports and for comparison with literature.

BSE plateau column: the convergence uncertainty of f_helix.
τ_slow ACF column: the ACF-derived relaxation time (cross-validates τ_eq).
"""
import pandas as pd

def _fmt(v, n=3):
    try:
        fv = float(v)
        return round(fv, n) if not np.isnan(fv) else 'N/A'
    except (TypeError, ValueError):
        return 'N/A'

rows = []
for key, r in results.items():
    s   = r['system']
    kin = r['kinetics']
    rows.append({
        'Peptide'          : s['prefix'],
        'Start'            : s['conf'],
        'Traj (µs)'        : _fmt(r['n_frames'] * DT_NS / 1000, 3),
        'f_helix'          : _fmt(kin['f_helix'], 3),
        'ΔG_fold (kcal)'   : _fmt(kin['dG_kcal'], 2),
        'N_fold'           : kin['n_fold'],
        'N_unfold'         : kin['n_unfold'],
        'τ_H (ns)'         : _fmt(kin['tau_H_ns'], 1),
        'τ_C (ns)'         : _fmt(kin['tau_C_ns'], 1),
        'k_f (ns⁻¹)'       : _fmt(kin['k_f_ns'], 4),
        'k_u (ns⁻¹)'       : _fmt(kin['k_u_ns'], 4),
        'τ_eq (ns)'        : _fmt(kin['tau_eq_ns'], 1),
        'τ_slow ACF (ns)'  : _fmt(r['tau_slow_ns'], 1),
        'BSE plateau'      : _fmt(r['bse_vals'][-1] if len(r['bse_vals']) else np.nan, 4),
    })

df = pd.DataFrame(rows)
print(df.to_string(index=False))
df.to_csv('kinetics_summary.csv', index=False)
print('\nSaved: kinetics_summary.csv')

print()
print('Literature reference — Kuczera & Jas 2021, ALA8, CHARMM36m, 300 K:')
print('  k_u = (7.0–7.9)×10⁻² ns⁻¹    k_f = (3.6–7.8)×10⁻³ ns⁻¹')
print('  ΔG_fold = 1.3–1.8 kcal/mol    τ_eq = 12–13 ns')
print()
print('Arrhenius correction 300 → 320 K (Eₐ ≈ 10 kcal/mol):')
import math
speedup = math.exp(10_000 / 1.987 * (1/300 - 1/320))
print(f'  Predicted speedup: {speedup:.1f}×')
print(f'  Expected τ_eq(320 K): {12.5 / speedup:.2f} – {12.0 / speedup:.2f} ns')

## Analysis Complete

### Output files generated

| File | Cell | Description |
|------|------|-------------|
| `{prefix}_{conf}_convergence.png` | 5 | Cumulative f_helix + BSE + split-trajectory FES |
| `{prefix}_{conf}_timeseries.png` | 6 | RMSD + N_HB + DSSP time series with state shading and transition markers |
| `{prefix}_{conf}_triple_criterion.png` | 7 | Three independent criteria + agreement score |
| `{prefix}_{conf}_populations.png` | 8 | Per-residue helix fraction (torsion) + per-bond H-bond occupancy + DSSP per-residue |
| `{prefix}_{conf}_FES.png` | 9 | 2D FES (RMSD × N_HB) + 1D FES along N_HB |
| `{prefix}_{conf}_Ramachandran.png` | 10 | Per-residue φ/ψ PMF (8° bins, 4 kcal/mol clip) |
| `{prefix}_{conf}_kinetics.png` | 11 | Sojourn distributions + ACF biexponential + literature comparison |
| `A10_vs_A8Q2_comparison.png` | 12 | Side-by-side FES + per-residue helix overlay + 1D FES comparison |
| `torsion_covariance_overlap.png` | 13 | PC spectrum + overlap matrix + diagonal similarity (CV-transfer verdict) |
| `kinetics_summary.csv` | 14 | Complete kinetic and thermodynamic summary table |

### Scientific validity checklist

| Check | Method | Source | Pass criterion |
|-------|--------|--------|----------------|
| Convergence (split-trajectory) | \|f₁ − f₂\| | Grossfield & Zuckerman 2009 | < 0.02 |
| Convergence (BSE plateau) | Block Standard Error | Grossfield & Zuckerman 2009 | < 0.01 |
| State assignment quality | Contested frames | Triple-criterion | < 10% |
| Kinetics cross-validation | τ_eq vs τ_slow(ACF) | Kuczera 2021 | Within factor 2 |
| Per-bond H-bond pattern | Central bonds > terminal | Baker-Hubbard 1984 | Central occupancy > 0.5 |
| Force field validation | k_f, k_u vs Kuczera 2021 (Arrhenius-corrected) | Kuczera 2021 Table 2 | Within 2–5× |
| CV-transfer readiness | PC-1, PC-2 eigenvector overlap | Pérez-Hernández 2013 | > 0.70 |

### Key references

1. **Kuczera & Jas (2021)** *Life* 11, 385 — ALA_n helix kinetics protocol; primary benchmark
2. **Grossfield & Zuckerman (2009)** *Annu. Rep. Comput. Chem.* 5 — BSE convergence methodology
3. **Tavernelli et al. (2003)** *Biophys. J.* 85 — Boltzmann inversion FES
4. **Best & Hummer (2009)** *J. Phys. Chem. B* 113 — Per-residue helix fraction definition
5. **Kabsch & Sander (1983)** *Biopolymers* 22 — DSSP algorithm
6. **Pérez-Hernández et al. (2013)** *J. Chem. Phys.* 139 — TICA torsion features and covariance